# Delay-scan coherence and gradient analysis

This notebook contains the final single-run pipeline for the independent 274 eV delay scans and the retained diagnostics used to investigate the delay-dependent spectral gradient. It uses lightweight time-resolved eTOF aggregates, the saved eTOF kinetic-energy calibration, GMD normalisation where specified, and shot-resolved combined HDF5 files for acquisition-order diagnostics.

Exploratory loader comparisons, RAM tests, preliminary Schwickert-style variants, and separate 2D-map experiments from the working notebook are omitted.

## Imports and environment

In [ ]:
import json
import os
import re
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress

%matplotlib inline

os.environ.setdefault("FLASH_ENV", "remote")


def find_repo_root(start=None):
    """Find the repository containing analysis/scripts."""
    candidates = []
    env_root = os.environ.get("GLYCINE_REPO_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())

    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates.extend([start, *start.parents])

    for candidate in candidates:
        if (candidate / "analysis" / "scripts").exists():
            return candidate
    raise RuntimeError(
        "Could not find the repository root. Run inside the repository or set "
        "GLYCINE_REPO_ROOT."
    )


repo_root = find_repo_root()
scripts_dir = repo_root / "analysis" / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import config
from compute_aggregates import load_aggregates

AggregatesData = load_aggregates.__globals__["AggregatesData"]


## Lightweight aggregate loading

Only the arrays required for the eTOF delay analysis are loaded. Large covariance matrices are deliberately omitted to keep memory use manageable.

In [ ]:
def load_aggregates_etof(path):
    """Load the eTOF aggregate arrays without the large covariance matrices."""
    path = Path(path)
    with h5py.File(path, "r") as handle:
        metadata = {}
        for key in handle.attrs:
            value = handle.attrs[key]
            metadata[key] = value.tolist() if isinstance(value, np.ndarray) else value

        def optional(key):
            return handle[key][:] if key in handle else None

        return AggregatesData(
            G=handle["G"][:],
            GtG=optional("GtG"),
            n_per_bin=handle["n_per_bin"][:],
            gmd_edges=handle["gmd_edges"][:],
            D=optional("D"),
            tof_edges=optional("tof_edges"),
            DtD=None,
            DtG=None,
            A=None,
            AtA=None,
            AtD=None,
            AtG=None,
            vls_pixels=optional("vls_pixels"),
            background=optional("background"),
            C=None,
            CtC=None,
            DtC=None,
            CtG=None,
            ion_tof_edges=optional("ion_tof_edges"),
            z_edges=optional("z_edges"),
            nominal_energies=optional("nominal_energies"),
            metadata=metadata,
        )


## Delay-scan inputs

In [ ]:
AGG_DIR = Path(config.COMBINED_DIR)
AGG_PATTERNS = [
    "glycine_delay_scan_7June_147C_*eV_aggregates_tr_etof1.h5",
    "glycine_delay_scan_150C_*eV_aggregates_tr_etof1.h5",
    "glycine_delay_scan_150C_long_*eV_aggregates_tr_etof1.h5",
    "glycine_delay_scan_160C_*eV_aggregates_tr_etof1.h5",
]
MIN_ENERGY = 272.0
MAX_ENERGY = 278.0
ENERGY_RE = re.compile(r"(\d+(?:\.\d+)?)eV")

aggregate_files = []
for pattern in AGG_PATTERNS:
    for path in sorted(AGG_DIR.glob(pattern)):
        match = ENERGY_RE.search(path.name)
        if match is None:
            continue
        energy = float(match.group(1))
        if MIN_ENERGY <= energy <= MAX_ENERGY:
            aggregate_files.append((energy, path))

aggregate_files = sorted(set(aggregate_files), key=lambda item: (item[0], item[1].name))
if not aggregate_files:
    raise FileNotFoundError(f"No delay-scan aggregates found in {AGG_DIR}")

loaded_aggs = {
    path.name: {
        "energy": energy,
        "path": path,
        "agg": load_aggregates_etof(path),
    }
    for energy, path in aggregate_files
}

print(
    f"Loaded {len(loaded_aggs)} lightweight delay aggregates from "
    f"{MIN_ENERGY:.1f} to {MAX_ENERGY:.1f} eV."
)


## eTOF calibration

The saved calibration is used without interpolation. KE-window integrals use fractional overlap with the native calibrated bins, so no Jacobian is needed for those integrated counts.

In [ ]:
CALIB_PATH = (
    Path(config.COMBINED_DIR).parent
    / "Calibrations"
    / "etof_energy_calibration_t0_scan.json"
)

with open(CALIB_PATH, "r", encoding="utf-8") as handle:
    calibration = json.load(handle)

best_t0 = calibration["t0_tof_units"]
best_slope = calibration["slope_eV_tof_unit2"]
best_E0 = calibration["E0_eV"]


def tof_to_ke(tof):
    """Apply the saved eTOF-to-electron-kinetic-energy calibration."""
    tof = np.asarray(tof, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        kinetic_energy = best_slope / (tof - best_t0) ** 2 + best_E0
    return np.where(tof > best_t0, kinetic_energy, np.nan)


## Single pipeline

Four 274 eV acquisitions are analysed independently. Each run is reconstructed from its aggregate means and shot counts, normalised by summed GMD, optionally corrected with the static residual-gas spectrum whose TOF edges match exactly, and evaluated in the broad and published narrow KE windows. The oscillation period is fixed at 19.6 fs; no smoothing, interpolation, or detrending is applied.

### Configuration and run selection

In [ ]:
# Settings

AGG_DIR = Path(config.COMBINED_DIR)
INCIDENT_ENERGY_EV = 274.0
EXPECTED_PERIOD_FS = 19.6

# Match the early-time range used for the coherence search.
FIT_RANGE_FS = (0.0, 25.0)

# Current delay conversion used in your pipeline.
DELAY_ZERO_STAGE = 0.0
FS_PER_STAGE_UNIT = 3.33 / 10000.0

# Primary normalisation.
# "per_uJ" here means: electrons in KE region / summed GMD
# exactly as in your existing pipeline.
NORMALISE = "per_uJ"

# Residual-gas treatment.
# Schwickert subtracts residual gas in the off-resonant analysis.
# Keep this True for the primary analysis.
# Because this is a STATIC background, it cannot remove a genuinely
# delay-dependent background contribution. It only subtracts the
# measured average residual-gas spectrum.
SUBTRACT_BACKGROUND = True
BACKGROUND_SCALE = 1.0

# Broad regions from the thesis-style diagnostic.
BROAD_LOW_WINDOWS_KE_EV = [
    (225.0, 240.0),
]
BROAD_HIGH_WINDOWS_KE_EV = [
    (245.0, 266.0),
]

# Published narrow regions from the off-resonant analysis.
# We SUM the two windows on each side before calculating relative
# changes. This makes better statistical use of the available counts.
NARROW_LOW_WINDOWS_KE_EV = [
    (231.0, 235.0),
    (235.0, 239.0),
]
NARROW_HIGH_WINDOWS_KE_EV = [
    (252.0, 256.0),
    (259.0, 263.0),
]

# Four independent 274 eV runs.
# The fourth file (7June_150C) was present on disk but was not in your
# earlier loaded_aggs list.
RUN_BASE_NAMES = [
    "glycine_delay_scan_150C_274.0eV",
    "glycine_delay_scan_160C_274.0eV",
    "glycine_delay_scan_7June_147C_274.0eV",
    "glycine_delay_scan_7June_150C_274.0eV",
]

RUN_LABELS = {
    "glycine_delay_scan_150C_274.0eV": "150C",
    "glycine_delay_scan_160C_274.0eV": "160C",
    "glycine_delay_scan_7June_147C_274.0eV": "7 June, 147C",
    "glycine_delay_scan_7June_150C_274.0eV": "7 June, 150C",
}

# Check calibration dependency
if "tof_to_ke" not in globals():
    raise NameError(
        "This cell requires your existing calibrated `tof_to_ke()` "
        "function. Run your eTOF calibration cell first."
    )

if not callable(tof_to_ke):
    raise TypeError("`tof_to_ke` exists but is not callable.")


### Reconstruction and fixed-period fitting helpers

In [ ]:
# Resolve aggregate filenames
def resolve_signal_aggregate(base_name):
    """Prefer _etof1, but also accept _etof1_5."""
    candidates = [
        AGG_DIR / f"{base_name}_aggregates_tr_etof1.h5",
        AGG_DIR / f"{base_name}_aggregates_tr_etof1_5.h5",
    ]
    existing = [p for p in candidates if p.exists()]

    if len(existing) == 0:
        raise FileNotFoundError(
            "Could not find an eTOF aggregate for:\n"
            f"  {base_name}\n\n"
            "Tried:\n" + "\n".join(f"  {p}" for p in candidates)
        )

    if len(existing) > 1:
        print("WARNING: both etof1 and etof1_5 exist for", base_name)
        print("Using:", existing[0].name)

    return existing[0]


signal_paths = {
    base_name: resolve_signal_aggregate(base_name) for base_name in RUN_BASE_NAMES
}

# Candidate 274 eV residual-gas aggregates
# There are two TOF-edge conventions:
#     etof1
#     etof1_5
# We DO NOT interpolate between them.
# For each signal run we select the background whose tof_edges match
# the signal tof_edges exactly.

BACKGROUND_DIR = AGG_DIR / "background_run58890_per_energy"

BACKGROUND_CANDIDATES = [
    BACKGROUND_DIR / "background_274.0eV_run58890_aggregates_etof1.h5",
    BACKGROUND_DIR / "background_274.0eV_run58890_aggregates_etof1_5.h5",
]

if SUBTRACT_BACKGROUND:

    missing_backgrounds = [p for p in BACKGROUND_CANDIDATES if not p.exists()]

    if len(missing_backgrounds) == len(BACKGROUND_CANDIDATES):
        raise FileNotFoundError(
            "Neither 274 eV residual-gas aggregate was found.\n"
            "Tried:\n" + "\n".join(f"  {p}" for p in BACKGROUND_CANDIDATES)
        )

# Use the existing lightweight aggregate loader.


def load_etof_aggregate_light(path):
    """Small adapter around the existing load_aggregates_etof()."""
    a = load_aggregates_etof(path)

    return {
        "path": Path(path),
        "D": np.asarray(a.D, dtype=float),
        "G": np.asarray(a.G, dtype=float),
        "n": np.asarray(a.n_per_bin, dtype=float),
        "tof_edges": np.asarray(a.tof_edges, dtype=float),
        "z_edges": None if a.z_edges is None else np.asarray(a.z_edges, dtype=float),
    }


# Select the residual-gas aggregate with EXACTLY matching TOF edges.
# No interpolation between etof1 and etof1_5 is performed.


def load_matching_background(signal_tof_edges):
    if not SUBTRACT_BACKGROUND:
        return None

    matching_path = None

    for bg_path in BACKGROUND_CANDIDATES:
        if not bg_path.exists():
            continue

        with h5py.File(bg_path, "r") as f:
            bg_tof_edges = np.asarray(f["tof_edges"][:], dtype=float)

        if np.array_equal(signal_tof_edges, bg_tof_edges):
            if matching_path is not None:
                raise ValueError(
                    "More than one background has identical TOF edges:\n"
                    f"  {matching_path.name}\n"
                    f"  {bg_path.name}"
                )

            matching_path = bg_path

    if matching_path is None:
        print("\nNo matching background TOF grid found.")
        print("Signal first 5 TOF edges:", signal_tof_edges[:5])

        for bg_path in BACKGROUND_CANDIDATES:
            if not bg_path.exists():
                continue

            with h5py.File(bg_path, "r") as f:
                bg_edges = np.asarray(f["tof_edges"][:], dtype=float)

            print(f"{bg_path.name}:")
            print("  first 5:", bg_edges[:5])

        raise ValueError(
            "Neither etof1 nor etof1_5 background has exactly "
            "the same TOF edges as this signal run."
        )

    return load_etof_aggregate_light(matching_path)


# KE integration helpers
def make_ke_window_weights(tof_edges, windows):
    """
    Generate fractional calibrated-KE-bin overlap weights for one or
    more disjoint KE windows. No interpolation and no smoothing are performed.
    """
    tof_edges = np.asarray(tof_edges, dtype=float)
    ke_edges = tof_to_ke(tof_edges)

    e0 = ke_edges[:-1]
    e1 = ke_edges[1:]
    bin_lo = np.minimum(e0, e1)
    bin_hi = np.maximum(e0, e1)
    widths = bin_hi - bin_lo

    valid = np.isfinite(e0) & np.isfinite(e1) & np.isfinite(widths) & (widths > 0)
    weights = np.zeros(len(tof_edges) - 1, dtype=float)

    for ke_min, ke_max in windows:
        overlap = np.maximum(
            0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min)
        )
        this_weight = np.zeros_like(weights)
        this_weight[valid] = overlap[valid] / widths[valid]
        weights += this_weight

    # Windows are disjoint here, but clip for safety.
    weights = np.clip(weights, 0.0, 1.0)

    if not np.any(weights > 0):
        raise ValueError(
            f"No calibrated TOF bins overlap requested KE windows: {windows}"
        )

    return weights


# Convert one aggregate into raw counts, shots and GMD versus delay
def reconstruct_signal_aggregate(data):
    D = data["D"]
    G = data["G"]
    n = data["n"]

    if D.ndim != 3:
        raise ValueError(
            f"{data['path'].name}: expected D[gmd, delay, tof], got {D.shape}"
        )
    if n.ndim != 2:
        raise ValueError(f"{data['path'].name}: expected n[gmd, delay], got {n.shape}")
    if G.shape != n.shape:
        raise ValueError(f"{data['path'].name}: G shape {G.shape} != n shape {n.shape}")
    if D.shape[:2] != n.shape:
        raise ValueError(
            f"{data['path'].name}: D leading shape {D.shape[:2]} != n shape {n.shape}"
        )

    # D = average number of electrons per shot in each aggregate bin.
    # Multiply by number of shots to reconstruct counts.
    counts_g_z_tof = np.nan_to_num(D * n[..., None], nan=0.0)
    counts_z_tof = np.sum(counts_g_z_tof, axis=0)
    shots_z = np.sum(np.nan_to_num(n, nan=0.0), axis=0)

    # G is the average GMD for each gmd/delay aggregate bin.
    # G*n therefore gives integrated GMD exposure.
    gmd_g_z = np.nan_to_num(G * n, nan=0.0)
    gmd_z = np.sum(gmd_g_z, axis=0)

    return counts_z_tof, shots_z, gmd_z


# Reconstruct static background
def reconstruct_background_aggregate(data):
    D = data["D"]
    G = data["G"]
    n = data["n"]

    # Static background: D[gmd, tof], G[gmd], n[gmd]
    if D.ndim == 2 and n.ndim == 1 and G.ndim == 1:
        if D.shape[0] != n.shape[0] or G.shape[0] != n.shape[0]:
            raise ValueError("Background GMD dimensions do not match.")
        counts_tof = np.sum(np.nan_to_num(D * n[:, None], nan=0.0), axis=0)
        shots = np.sum(np.nan_to_num(n, nan=0.0))
        gmd = np.sum(np.nan_to_num(G * n, nan=0.0))

    # Also allow time-resolved-shaped background.
    elif D.ndim == 3 and n.ndim == 2 and G.ndim == 2:
        if D.shape[:2] != n.shape or G.shape != n.shape:
            raise ValueError("Background aggregate dimensions do not match.")
        counts_tof = np.sum(np.nan_to_num(D * n[..., None], nan=0.0), axis=(0, 1))
        shots = np.sum(np.nan_to_num(n, nan=0.0))
        gmd = np.sum(np.nan_to_num(G * n, nan=0.0))
    else:
        raise ValueError(
            f"Unexpected background shapes:\n  D = {D.shape}\n  G = {G.shape}\n  n = {n.shape}"
        )

    return counts_tof, shots, gmd


# Integrate one set of KE windows and normalise by GMD
def integrate_windows_per_uJ(counts_z_tof, gmd_z, tof_edges, windows, background):
    """Integrate one or more KE windows and return the GMD-normalised background-subtracted yield."""
    weights = make_ke_window_weights(tof_edges, windows)
    C_sig = np.sum(counts_z_tof * weights[None, :], axis=1)

    # Fractional-bin Poisson variance approximation.
    var_C_sig = np.sum(counts_z_tof * weights[None, :] ** 2, axis=1)

    with np.errstate(invalid="ignore", divide="ignore"):
        Y_sig = C_sig / gmd_z
        sigma_sig = np.sqrt(var_C_sig) / gmd_z

    if SUBTRACT_BACKGROUND:
        if background is None:
            raise ValueError(
                "SUBTRACT_BACKGROUND=True but no matching background was supplied."
            )

        if not np.array_equal(tof_edges, background["tof_edges"]):
            raise ValueError(
                "Internal error: selected background does not match signal TOF edges."
            )

        bg_counts_tof, bg_shots, bg_gmd = reconstruct_background_aggregate(background)

        if not np.isfinite(bg_gmd) or bg_gmd <= 0:
            raise ValueError("Background integrated GMD is invalid.")

        C_bg = np.sum(bg_counts_tof * weights)
        var_C_bg = np.sum(bg_counts_tof * weights**2)

        Y_bg = C_bg / bg_gmd
        sigma_bg = np.sqrt(var_C_bg) / bg_gmd

    else:
        Y_bg = 0.0
        sigma_bg = 0.0

    Y = Y_sig - BACKGROUND_SCALE * Y_bg
    sigma_Y = np.sqrt(sigma_sig**2 + (BACKGROUND_SCALE * sigma_bg) ** 2)

    return {
        "counts": C_sig,
        "yield": Y,
        "yield_error": sigma_Y,
        "weights": weights,
    }


# Convert absolute yield to relative change around its own delay average
def make_relative_trace(Y, sigma_Y):
    Y = np.asarray(Y, dtype=float)
    sigma_Y = np.asarray(sigma_Y, dtype=float)
    mean_Y = np.nanmean(Y)

    if not np.isfinite(mean_Y) or mean_Y == 0:
        raise ValueError("Mean integrated yield is zero or invalid.")

    relative_percent = 100.0 * (Y - mean_Y) / mean_Y
    relative_error_percent = 100.0 * sigma_Y / abs(mean_Y)

    return relative_percent, relative_error_percent, mean_Y


# Fixed-period fit
# y(t) = c + a sin(w t) + b cos(w t) = c + A sin(w t + phi)
# phi = atan2(b, a)
def fixed_period_fit(
    delay_fs,
    y_percent,
    sigma_percent,
    period_fs=EXPECTED_PERIOD_FS,
    fit_range_fs=FIT_RANGE_FS,
):
    delay_fs = np.asarray(delay_fs, dtype=float)
    y_percent = np.asarray(y_percent, dtype=float)
    sigma_percent = np.asarray(sigma_percent, dtype=float)

    fit_mask = (
        np.isfinite(delay_fs)
        & np.isfinite(y_percent)
        & (delay_fs >= fit_range_fs[0])
        & (delay_fs <= fit_range_fs[1])
    )

    if np.count_nonzero(fit_mask) < 4:
        raise ValueError(
            f"Fewer than four valid points are inside FIT_RANGE_FS={fit_range_fs}."
        )

    t = delay_fs[fit_mask]
    y = y_percent[fit_mask]
    sigma = sigma_percent[fit_mask]

    omega = 2.0 * np.pi / period_fs
    X = np.column_stack([np.ones_like(t), np.sin(omega * t), np.cos(omega * t)])
    valid_sigma = np.isfinite(sigma) & (sigma > 0)

    if np.count_nonzero(valid_sigma) >= 4:
        y_used = y[valid_sigma]
        sigma_used = sigma[valid_sigma]
        X_used = X[valid_sigma]

        sqrt_w = 1.0 / sigma_used
        Xw = X_used * sqrt_w[:, None]
        yw = y_used * sqrt_w

        coeff, _, _, _ = np.linalg.lstsq(Xw, yw, rcond=None)
        residual = y_used - X_used @ coeff
        dof = max(len(y_used) - len(coeff), 1)
        chi2 = np.sum((residual / sigma_used) ** 2)
        reduced_chi2 = chi2 / dof

        try:
            covariance = np.linalg.inv(Xw.T @ Xw)
            covariance *= reduced_chi2  # Scale uncertainties based on reduced chi2
        except np.linalg.LinAlgError:
            covariance = np.full((3, 3), np.nan)
    else:
        print(
            "WARNING: insufficient valid uncertainties; using unweighted fixed-period fit."
        )
        coeff, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        residual = y - X @ coeff
        dof = max(len(y) - len(coeff), 1)
        residual_variance = np.sum(residual**2) / dof
        reduced_chi2 = np.nan

        try:
            covariance = np.linalg.inv(X.T @ X) * residual_variance
        except np.linalg.LinAlgError:
            covariance = np.full((3, 3), np.nan)

    offset = coeff[0]
    sin_coeff = coeff[1]
    cos_coeff = coeff[2]

    amplitude = np.hypot(sin_coeff, cos_coeff)
    phase_rad = np.arctan2(cos_coeff, sin_coeff)
    phase_rad = np.angle(np.exp(1j * phase_rad))  # Wrap phase to [-pi, pi]

    # Approximate amplitude and phase uncertainties.
    if amplitude > 0 and np.all(np.isfinite(covariance[1:3, 1:3])):
        cov_ab = covariance[1:3, 1:3]
        grad_amp = np.array([sin_coeff / amplitude, cos_coeff / amplitude])
        var_amp = grad_amp @ cov_ab @ grad_amp

        grad_phase = np.array([-cos_coeff / amplitude**2, sin_coeff / amplitude**2])
        var_phase = grad_phase @ cov_ab @ grad_phase

        amplitude_error = np.sqrt(max(var_amp, 0.0))
        phase_error_rad = np.sqrt(max(var_phase, 0.0))
    else:
        amplitude_error = np.nan
        phase_error_rad = np.nan

    return {
        "period_fs": period_fs,
        "omega": omega,
        "offset_percent": offset,
        "amplitude_percent": amplitude,
        "amplitude_error_percent": amplitude_error,
        "phase_rad": phase_rad,
        "phase_error_rad": phase_error_rad,
        "phase_pi": phase_rad / np.pi,
        "phase_error_pi": phase_error_rad / np.pi,
        "reduced_chi2": reduced_chi2,
        "fit_mask": fit_mask,
    }


### Analyse the four independent runs

In [ ]:
# Process one complete run
def analyse_run(base_name, path):
    data = load_etof_aggregate_light(path)
    background = load_matching_background(data["tof_edges"])

    if background is not None:
        print("  matched background:", background["path"].name)

    if data["z_edges"] is None:
        raise KeyError(f"{path.name}: no z_edges dataset found.")

    counts_z_tof, shots_z, gmd_z = reconstruct_signal_aggregate(data)
    if counts_z_tof.shape[0] != (len(data["z_edges"]) - 1):
        raise ValueError(f"{path.name}: delay dimension does not match z_edges.")
    if not np.any(gmd_z > 0):
        raise ValueError(f"{path.name}: no positive GMD exposure.")

    z_cent = 0.5 * (data["z_edges"][:-1] + data["z_edges"][1:])
    delay_fs = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

    # BROAD
    broad_low = integrate_windows_per_uJ(
        counts_z_tof, gmd_z, data["tof_edges"], BROAD_LOW_WINDOWS_KE_EV, background
    )

    broad_high = integrate_windows_per_uJ(
        counts_z_tof, gmd_z, data["tof_edges"], BROAD_HIGH_WINDOWS_KE_EV, background
    )

    broad_low_rel, broad_low_err, _ = make_relative_trace(
        broad_low["yield"], broad_low["yield_error"]
    )
    broad_high_rel, broad_high_err, _ = make_relative_trace(
        broad_high["yield"], broad_high["yield_error"]
    )

    broad_difference = broad_high_rel - broad_low_rel
    broad_difference_error = np.sqrt(broad_high_err**2 + broad_low_err**2)
    broad_fit = fixed_period_fit(delay_fs, broad_difference, broad_difference_error)

    # PUBLISHED NARROW WINDOWS
    narrow_low = integrate_windows_per_uJ(
        counts_z_tof, gmd_z, data["tof_edges"], NARROW_LOW_WINDOWS_KE_EV, background
    )

    narrow_high = integrate_windows_per_uJ(
        counts_z_tof, gmd_z, data["tof_edges"], NARROW_HIGH_WINDOWS_KE_EV, background
    )

    narrow_low_rel, narrow_low_err, _ = make_relative_trace(
        narrow_low["yield"], narrow_low["yield_error"]
    )
    narrow_high_rel, narrow_high_err, _ = make_relative_trace(
        narrow_high["yield"], narrow_high["yield_error"]
    )

    narrow_difference = narrow_high_rel - narrow_low_rel
    narrow_difference_error = np.sqrt(narrow_high_err**2 + narrow_low_err**2)
    narrow_fit = fixed_period_fit(delay_fs, narrow_difference, narrow_difference_error)

    return {
        "base_name": base_name,
        "label": RUN_LABELS[base_name],
        "path": path,
        "tof_edges": data["tof_edges"],
        "delay_fs": delay_fs,
        "shots_z": shots_z,
        "gmd_z": gmd_z,
        "total_shots": np.nansum(shots_z),
        "total_gmd": np.nansum(gmd_z),
        # Broad
        "broad_low_rel": broad_low_rel,
        "broad_low_err": broad_low_err,
        "broad_high_rel": broad_high_rel,
        "broad_high_err": broad_high_err,
        "broad_difference": broad_difference,
        "broad_difference_error": broad_difference_error,
        "broad_fit": broad_fit,
        # Narrow
        "narrow_low_rel": narrow_low_rel,
        "narrow_low_err": narrow_low_err,
        "narrow_high_rel": narrow_high_rel,
        "narrow_high_err": narrow_high_err,
        "narrow_difference": narrow_difference,
        "narrow_difference_error": narrow_difference_error,
        "narrow_fit": narrow_fit,
    }


# Load and analyse all four runs
results = []
print("274 eV independent-run coherence search")
print("========================================")
print()
print("Normalisation:", NORMALISE)
print("Residual-gas subtraction:", SUBTRACT_BACKGROUND)
print("Fixed period:", EXPECTED_PERIOD_FS, "fs")
print("Fit range:", FIT_RANGE_FS, "fs")
print()

for base_name in RUN_BASE_NAMES:
    path = signal_paths[base_name]
    print("Loading:", path.name)
    result = analyse_run(base_name, path)
    results.append(result)

    print("  label:", result["label"])
    print("  total shots:", int(result["total_shots"]))
    print("  delay bins:", len(result["delay_fs"]))
    print(
        "  delay range:",
        f"{np.nanmin(result['delay_fs']):.3f} -> {np.nanmax(result['delay_fs']):.3f} fs",
    )
    print()


### Coherence-search figures and fit summaries

In [ ]:
# FIGURE 1: BROAD high / low traces and literal high-minus-low
fig, axes = plt.subplots(
    2, 2, figsize=(12, 8), sharex=True, sharey=True, constrained_layout=True
)
axes = axes.ravel()

for ax, result in zip(axes, results):
    t = result["delay_fs"]
    ax.errorbar(
        t,
        result["broad_low_rel"],
        yerr=result["broad_low_err"],
        fmt="o-",
        ms=3,
        lw=0.8,
        elinewidth=0.6,
        capsize=1.5,
        alpha=0.75,
        label="low 225-240 eV",
    )
    ax.errorbar(
        t,
        result["broad_high_rel"],
        yerr=result["broad_high_err"],
        fmt="s-",
        ms=3,
        lw=0.8,
        elinewidth=0.6,
        capsize=1.5,
        alpha=0.75,
        label="high 245-266 eV",
    )
    ax.errorbar(
        t,
        result["broad_difference"],
        yerr=result["broad_difference_error"],
        fmt="o",
        ms=3.5,
        elinewidth=0.7,
        capsize=1.5,
        label="high - low",
    )

    fit = result["broad_fit"]
    t_dense = np.linspace(FIT_RANGE_FS[0], FIT_RANGE_FS[1], 500)
    fit_dense = fit["offset_percent"] + fit["amplitude_percent"] * np.sin(
        fit["omega"] * t_dense + fit["phase_rad"]
    )

    ax.plot(t_dense, fit_dense, lw=1.6, label=f"T = {EXPECTED_PERIOD_FS:.1f} fs fit")
    ax.axhline(0.0, color="k", lw=0.7, alpha=0.5)
    ax.axvline(0.0, color="k", lw=0.7, alpha=0.5)
    ax.set_title(
        f"{INCIDENT_ENERGY_EV:.1f} eV - {result['label']}\nA = {fit['amplitude_percent']:.2f}%, phi = {fit['phase_pi']:+.2f}pi"
    )
    ax.set_xlabel("delay (fs)")
    ax.set_ylabel("relative electron yield (%)")
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
fig.suptitle("Broad KE regions: literal high - low\nGMD norm")

# FIGURE 2: PUBLISHED narrow windows and literal high-minus-low
fig, axes = plt.subplots(
    2, 2, figsize=(12, 8), sharex=True, sharey=True, constrained_layout=True
)
axes = axes.ravel()

for ax, result in zip(axes, results):
    t = result["delay_fs"]

    ax.errorbar(
        t,
        result["narrow_low_rel"],
        yerr=result["narrow_low_err"],
        fmt="o-",
        ms=3,
        lw=0.8,
        elinewidth=0.6,
        capsize=1.5,
        alpha=0.75,
        label="low: 231-235 + 235-239 eV",
    )

    ax.errorbar(
        t,
        result["narrow_high_rel"],
        yerr=result["narrow_high_err"],
        fmt="s-",
        ms=3,
        lw=0.8,
        elinewidth=0.6,
        capsize=1.5,
        alpha=0.75,
        label="high: 252-256 + 259-263 eV",
    )

    ax.errorbar(
        t,
        result["narrow_difference"],
        yerr=result["narrow_difference_error"],
        fmt="o",
        ms=3.5,
        elinewidth=0.7,
        capsize=1.5,
        label="high - low",
    )

    fit = result["narrow_fit"]
    t_dense = np.linspace(FIT_RANGE_FS[0], FIT_RANGE_FS[1], 500)
    fit_dense = fit["offset_percent"] + fit["amplitude_percent"] * np.sin(
        fit["omega"] * t_dense + fit["phase_rad"]
    )

    ax.plot(t_dense, fit_dense, lw=1.6, label=f"T = {EXPECTED_PERIOD_FS:.1f} fs fit")
    ax.axhline(0.0, color="k", lw=0.7, alpha=0.5)
    ax.axvline(0.0, color="k", lw=0.7, alpha=0.5)

    ax.set_title(
        f"{INCIDENT_ENERGY_EV:.1f} eV - {result['label']}\n"
        f"A = {fit['amplitude_percent']:.2f}%, phi = {fit['phase_pi']:+.2f}pi"
    )
    ax.set_xlabel("delay (fs)")
    ax.set_ylabel("relative electron yield (%)")
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
fig.suptitle(
    f"Published narrow KE regions: literal high - low\nGMD normalisation, fixed T = {EXPECTED_PERIOD_FS:.1f} fs",
    fontsize=13,
)
plt.show()

# Numerical fit summary
print()
print("FIXED-19.6-fs FIT RESULTS")
print("=========================")
print()
print("Phases refer to:")
print("    high-relative-yield - low-relative-yield")
print("with model:")
print("    c + A sin(2pit/T + phi)")
print()

for result in results:
    broad_fit = result["broad_fit"]
    narrow_fit = result["narrow_fit"]

    print(result["label"])
    print("  BROAD:")
    print(
        f"    amplitude = {broad_fit['amplitude_percent']:.3f} +/- {broad_fit['amplitude_error_percent']:.3f} %"
    )
    print(
        f"    phase     = {broad_fit['phase_rad']:+.3f} +/- {broad_fit['phase_error_rad']:.3f} rad"
    )
    print(
        f"              = {broad_fit['phase_pi']:+.3f} +/- {broad_fit['phase_error_pi']:.3f} pi"
    )
    print(f"    reduced chi^2 = {broad_fit['reduced_chi2']:.3f}")

    print("  NARROW:")
    print(
        f"    amplitude = {narrow_fit['amplitude_percent']:.3f} +/- {narrow_fit['amplitude_error_percent']:.3f} %"
    )
    print(
        f"    phase     = {narrow_fit['phase_rad']:+.3f} +/- {narrow_fit['phase_error_rad']:.3f} rad"
    )
    print(
        f"              = {narrow_fit['phase_pi']:+.3f} +/- {narrow_fit['phase_error_pi']:.3f} pi"
    )
    print(f"    reduced chi^2 = {narrow_fit['reduced_chi2']:.3f}")
    print()


# Phase comparison helper
# Calculate shortest phase difference modulo 2pi.
def circular_phase_difference(phi_a, phi_b):
    return np.angle(np.exp(1j * (phi_a - phi_b)))


# Pairwise phase-difference matrices
labels = [result["label"] for result in results]
broad_phases = np.array([result["broad_fit"]["phase_rad"] for result in results])
narrow_phases = np.array([result["narrow_fit"]["phase_rad"] for result in results])


def print_phase_matrix(phases, title):
    n_runs = len(phases)
    print()
    print(title)
    print("-" * len(title))

    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            dphi = circular_phase_difference(phases[j], phases[i])
            dphase_pi = dphi / np.pi
            equivalent_dt_fs = dphi / (2.0 * np.pi / EXPECTED_PERIOD_FS)
            print(
                f"{labels[i]:16s} -> {labels[j]:16s}: Deltaphi = {dphase_pi:+.3f} pi   equivalent Deltat = {equivalent_dt_fs:+.2f} fs"
            )


print_phase_matrix(broad_phases, "PAIRWISE BROAD PHASE DIFFERENCES")
print_phase_matrix(narrow_phases, "PAIRWISE NARROW PHASE DIFFERENCES")


# Circular phase concentration
# R = 1   -> phases identical
# R ~ 0   -> phases spread around the circle
# Diagnostic only. It is NOT a statistical significance test.
def circular_resultant_length(phases):
    return np.abs(np.mean(np.exp(1j * phases)))


broad_R = circular_resultant_length(broad_phases)
narrow_R = circular_resultant_length(narrow_phases)

print()
print("PHASE CONSISTENCY DIAGNOSTIC")
print("----------------------------")
print(f"Broad-window circular phase concentration R = {broad_R:.3f}")
print(f"Narrow-window circular phase concentration R = {narrow_R:.3f}")
print("(R = 1 means identical phases; R near 0 means widely distributed phases.)")

# FIGURE 3: phase + amplitude comparison
x = np.arange(len(results))
broad_phase_pi = np.array([result["broad_fit"]["phase_pi"] for result in results])
broad_phase_err_pi = np.array(
    [result["broad_fit"]["phase_error_pi"] for result in results]
)
narrow_phase_pi = np.array([result["narrow_fit"]["phase_pi"] for result in results])
narrow_phase_err_pi = np.array(
    [result["narrow_fit"]["phase_error_pi"] for result in results]
)

broad_amp = np.array([result["broad_fit"]["amplitude_percent"] for result in results])
broad_amp_err = np.array(
    [result["broad_fit"]["amplitude_error_percent"] for result in results]
)
narrow_amp = np.array([result["narrow_fit"]["amplitude_percent"] for result in results])
narrow_amp_err = np.array(
    [result["narrow_fit"]["amplitude_error_percent"] for result in results]
)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True, constrained_layout=True)

# Phase
axes[0].errorbar(
    x - 0.08,
    broad_phase_pi,
    yerr=broad_phase_err_pi,
    fmt="o",
    ms=6,
    capsize=3,
    label="broad windows",
)
axes[0].errorbar(
    x + 0.08,
    narrow_phase_pi,
    yerr=narrow_phase_err_pi,
    fmt="s",
    ms=6,
    capsize=3,
    label="published narrow windows",
)
axes[0].axhline(0.0, color="k", lw=0.7, alpha=0.4)
axes[0].set_ylabel("fitted phase / pi")
axes[0].set_title(f"Fixed-{EXPECTED_PERIOD_FS:.1f}-fs phase comparison")
axes[0].legend()
axes[0].grid(alpha=0.2)

# Amplitude
axes[1].errorbar(
    x - 0.08,
    broad_amp,
    yerr=broad_amp_err,
    fmt="o",
    ms=6,
    capsize=3,
    label="broad windows",
)
axes[1].errorbar(
    x + 0.08,
    narrow_amp,
    yerr=narrow_amp_err,
    fmt="s",
    ms=6,
    capsize=3,
    label="published narrow windows",
)
axes[1].set_ylabel("high - low amplitude (%)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=20, ha="right")
axes[1].set_xlabel("independent 274 eV acquisition")
axes[1].grid(alpha=0.2)

plt.show()


## Investigating the gradient with delay

The following diagnostics test whether the observed redistribution is stable across photon energies, GMD bins, repeated acquisitions, and kinetic-energy regions, and whether it correlates with acquisition variables or low-energy residual-gas structure. Unless a cell states otherwise, these diagnostics use no smoothing, interpolation, detrending, Jacobian, or residual-gas subtraction.

### Delay-dependent spectral slope at one photon energy

In [ ]:
# User choices & bounds
DIAG_ENERGY_EV = 275.0
LOW_WINDOW_KE_EV, HIGH_WINDOW_KE_EV = (225.0, 240.0), (245.0, 266.0)
DIAG_KE_RANGE_EV = (220.0, 270.0)
DELAY_ROUND_DECIMALS = 6


def _centres_to_edges(x):
    x = np.asarray(x, dtype=float)
    if x.size < 2:
        raise ValueError("Need at least two delay centres to construct edges.")
    edges = np.empty(x.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (x[:-1] + x[1:])
    edges[0], edges[-1] = x[0] - 0.5 * (x[1] - x[0]), x[-1] + 0.5 * (x[-1] - x[-2])
    return edges


def _ke_window_weights(tof_edges, ke_min, ke_max):
    ke_edges = tof_to_ke(np.asarray(tof_edges, dtype=float))
    e0, e1 = ke_edges[:-1], ke_edges[1:]
    valid = np.isfinite(e0) & np.isfinite(e1)
    bin_lo, bin_hi = np.minimum(e0, e1), np.maximum(e0, e1)
    bin_width = bin_hi - bin_lo
    overlap = np.maximum(0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min))
    weights = np.zeros_like(bin_width, dtype=float)
    good = valid & (bin_width > 0)
    weights[good] = overlap[good] / bin_width[good]
    return weights


def _relative_change(y):
    y = np.asarray(y, dtype=float)
    mean = np.nanmean(y)
    with np.errstate(invalid="ignore", divide="ignore"):
        return 100.0 * (y - mean) / mean


# Select and validate loaded files
matching = [
    (key, item)
    for key, item in loaded_aggs.items()
    if np.isclose(item["energy"], DIAG_ENERGY_EV)
]
if not matching:
    raise ValueError(f"No loaded aggregate files found at {DIAG_ENERGY_EV:.1f} eV.")

print(f"Combining {len(matching)} file(s) at {DIAG_ENERGY_EV:.1f} eV:")
for key, item in matching:
    print("  ", key)

tof_edges_ref, gmd_edges_ref = None, None
for key, item in matching:
    a = item["agg"]
    if tof_edges_ref is None:
        tof_edges_ref, gmd_edges_ref = np.asarray(a.tof_edges, dtype=float), np.asarray(
            a.gmd_edges, dtype=float
        )
    else:
        if not np.array_equal(tof_edges_ref, a.tof_edges):
            raise ValueError(f"{key}: TOF edges differ.")
        if not np.array_equal(gmd_edges_ref, a.gmd_edges):
            raise ValueError(f"{key}: GMD edges differ.")

# Merge physical delay centres retaining GMD dimensions
counts_by_delay, shots_by_delay, gmd_by_delay = {}, {}, {}
for key, item in matching:
    a = item["agg"]
    D, G, n = (
        np.asarray(a.D, dtype=float),
        np.asarray(a.G, dtype=float),
        np.asarray(a.n_per_bin, dtype=float),
    )

    if (
        D.ndim != 3
        or D.shape[:2] != n.shape
        or G.shape != n.shape
        or D.shape[2] != len(tof_edges_ref) - 1
    ):
        raise ValueError(f"{key}: Aggregate array dimension mismatch.")

    delay_fs = (
        0.5 * (a.z_edges[:-1] + a.z_edges[1:]) - DELAY_ZERO_STAGE
    ) * FS_PER_STAGE_UNIT
    counts_g_z_tof, gmdsum_g_z = D * n[..., None], G * n

    for iz, delay in enumerate(delay_fs):
        dk = round(float(delay), DELAY_ROUND_DECIMALS)
        if dk not in counts_by_delay:
            counts_by_delay[dk] = np.zeros((D.shape[0], D.shape[2]), dtype=float)
            shots_by_delay[dk] = np.zeros(D.shape[0], dtype=float)
            gmd_by_delay[dk] = np.zeros(D.shape[0], dtype=float)
        counts_by_delay[dk] += np.nan_to_num(counts_g_z_tof[:, iz, :], nan=0.0)
        shots_by_delay[dk] += np.nan_to_num(n[:, iz], nan=0.0)
        gmd_by_delay[dk] += np.nan_to_num(gmdsum_g_z[:, iz], nan=0.0)

# Build unified arrays
delay_cent_fs_diag = np.array(sorted(counts_by_delay), dtype=float)
counts_g_z_tof = np.stack([counts_by_delay[t] for t in delay_cent_fs_diag], axis=1)
shots_g_z = np.stack([shots_by_delay[t] for t in delay_cent_fs_diag], axis=1)
gmdsum_g_z = np.stack([gmd_by_delay[t] for t in delay_cent_fs_diag], axis=1)

print(
    f"\nMerged shapes:\ncounts_g_z_tof: {counts_g_z_tof.shape}\nshots_g_z: {shots_g_z.shape}\ngmdsum_g_z: {gmdsum_g_z.shape}\n"
)

counts_z_tof = np.nansum(counts_g_z_tof, axis=0)
shots_z, gmdsum_z = np.nansum(shots_g_z, axis=0), np.nansum(gmdsum_g_z, axis=0)
total_electrons_z = np.nansum(counts_z_tof, axis=1)
with np.errstate(invalid="ignore", divide="ignore"):
    mean_gmd_per_shot_z = gmdsum_z / shots_z

# Coordinate calibration and masks
tof_edges = np.asarray(tof_edges_ref, dtype=float)
tof_cent, ke_edges = 0.5 * (tof_edges[:-1] + tof_edges[1:]), tof_to_ke(tof_edges)
ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])
diag_ke_mask = (
    np.isfinite(ke_edges[:-1])
    & np.isfinite(ke_edges[1:])
    & np.isfinite(ke_cent)
    & (ke_cent >= DIAG_KE_RANGE_EV[0])
    & (ke_cent <= DIAG_KE_RANGE_EV[1])
)
if not np.any(diag_ke_mask):
    raise ValueError(f"No bins lie inside {DIAG_KE_RANGE_EV} eV.")

# Extract Centroids
counts_centroid = counts_z_tof[:, diag_ke_mask]
centroid_den = np.nansum(counts_centroid, axis=1)
with np.errstate(invalid="ignore", divide="ignore"):
    ke_centroid_z = (
        np.nansum(counts_centroid * ke_cent[diag_ke_mask][None, :], axis=1)
        / centroid_den
    )
    tof_centroid_z = (
        np.nansum(counts_centroid * tof_cent[diag_ke_mask][None, :], axis=1)
        / centroid_den
    )

# Integrated Yield Regions
w_low = _ke_window_weights(tof_edges, LOW_WINDOW_KE_EV[0], LOW_WINDOW_KE_EV[1])
w_high = _ke_window_weights(tof_edges, HIGH_WINDOW_KE_EV[0], HIGH_WINDOW_KE_EV[1])
w_diag = _ke_window_weights(tof_edges, DIAG_KE_RANGE_EV[0], DIAG_KE_RANGE_EV[1])
w_other = np.clip(w_diag - w_low - w_high, 0.0, 1.0)

with np.errstate(invalid="ignore", divide="ignore"):
    low_rel = _relative_change(
        np.nansum(counts_z_tof * w_low[None, :], axis=1) / total_electrons_z
    )
    high_rel = _relative_change(
        np.nansum(counts_z_tof * w_high[None, :], axis=1) / total_electrons_z
    )
    other_rel = _relative_change(
        np.nansum(counts_z_tof * w_other[None, :], axis=1) / total_electrons_z
    )
    outside_rel = _relative_change(
        (total_electrons_z - np.nansum(counts_z_tof * w_diag[None, :], axis=1))
        / total_electrons_z
    )

with np.errstate(invalid="ignore", divide="ignore"):
    gmd_shot_fraction = shots_g_z / np.nansum(shots_g_z, axis=0)[None, :]
gmd_labels = [
    f"{gmd_edges_ref[i]:.3g}-{gmd_edges_ref[i+1]:.3g}"
    for i in range(len(gmd_edges_ref) - 1)
]

# FIGURE 1: Scalar Diagnostics
fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True, constrained_layout=True)
axes = axes.ravel()

axes[0].plot(delay_cent_fs_diag, ke_centroid_z, "o-", ms=3, lw=1.0)
axes[0].set_ylabel("KE centroid (eV)")
axes[0].set_title(
    f"Spectral centroid, {DIAG_KE_RANGE_EV[0]:.0f}-{DIAG_KE_RANGE_EV[1]:.0f} eV"
)

axes[1].plot(delay_cent_fs_diag, tof_centroid_z, "o-", ms=3, lw=1.0)
axes[1].set_ylabel("TOF centroid")
axes[1].set_title("Same centroid in raw TOF coordinates")

axes[2].plot(
    delay_cent_fs_diag,
    low_rel,
    "o-",
    ms=3,
    lw=1.0,
    label=f"low {LOW_WINDOW_KE_EV[0]:.0f}-{LOW_WINDOW_KE_EV[1]:.0f} eV",
)
axes[2].plot(
    delay_cent_fs_diag,
    high_rel,
    "o-",
    ms=3,
    lw=1.0,
    label=f"high {HIGH_WINDOW_KE_EV[0]:.0f}-{HIGH_WINDOW_KE_EV[1]:.0f} eV",
)
axes[2].plot(
    delay_cent_fs_diag,
    other_rel,
    "o-",
    ms=3,
    lw=1.0,
    label=f"other within {DIAG_KE_RANGE_EV[0]:.0f}-{DIAG_KE_RANGE_EV[1]:.0f} eV",
)
axes[2].plot(
    delay_cent_fs_diag,
    outside_rel,
    "o-",
    ms=3,
    lw=1.0,
    label="outside diagnostic KE range",
)
axes[2].axhline(0.0, color="k", lw=0.8, alpha=0.5)
axes[2].set_ylabel("relative electron yield (%)")
axes[2].set_title("Where is the normalized yield moving?")
axes[2].legend(fontsize=8)

for ig, label in enumerate(gmd_labels):
    axes[3].plot(
        delay_cent_fs_diag,
        100.0 * gmd_shot_fraction[ig],
        "o-",
        ms=3,
        lw=1.0,
        label=label,
    )
axes[3].set_ylabel("shots in GMD bin (%)")
axes[3].set_title("GMD-bin population vs delay")
axes[3].legend(title="GMD bin", fontsize=8)

axes[4].plot(delay_cent_fs_diag, total_electrons_z, "o-", ms=3, lw=1.0)
axes[4].set_ylabel("total detected electrons")
axes[4].set_title("Total electron yield")

ax5b = axes[5].twinx()
line1 = axes[5].plot(
    delay_cent_fs_diag, gmdsum_z, "o-", ms=3, lw=1.0, label="summed GMD"
)
line2 = ax5b.plot(
    delay_cent_fs_diag,
    mean_gmd_per_shot_z,
    "s-",
    ms=3,
    lw=1.0,
    label="mean GMD / shot",
    color="orange",
)
axes[5].set_ylabel("summed GMD")
ax5b.set_ylabel("mean GMD / shot")
axes[5].set_title("GMD diagnostics")
lines = line1 + line2
axes[5].legend(lines, [ln.get_label() for ln in lines], fontsize=8)

for ax in axes:
    ax.set_xlabel("delay (fs)")
    ax.grid(alpha=0.2)
fig.suptitle(
    f"{DIAG_ENERGY_EV:.1f} eV delay-scan diagnostics ({len(matching)} file{'s' if len(matching) != 1 else ''} merged)\nNo residual-gas subtraction, no detrending",
    fontsize=13,
)
plt.show()

# FIGURE 2: 2D Relative Change Maps
with np.errstate(invalid="ignore", divide="ignore"):
    spectrum_fraction_z_tof = counts_z_tof / total_electrons_z[:, None]
avg_spectrum_fraction = np.nanmean(spectrum_fraction_z_tof, axis=0)
with np.errstate(invalid="ignore", divide="ignore"):
    rel_map_z_tof = (
        100.0
        * (spectrum_fraction_z_tof - avg_spectrum_fraction[None, :])
        / avg_spectrum_fraction[None, :]
    )

delay_edges_diag = _centres_to_edges(delay_cent_fs_diag)
diag_idx = np.where(diag_ke_mask)[0]
lo_diag, hi_diag = int(diag_idx[0]), int(diag_idx[-1]) + 1

rel_map_crop = rel_map_z_tof[:, lo_diag:hi_diag]
tof_edges_crop = tof_edges[lo_diag : hi_diag + 1]
ke_edges_crop = ke_edges[lo_diag : hi_diag + 1]
vmax = 30 if rel_map_crop[np.isfinite(rel_map_crop)].size else 10.0

fig, axes = plt.subplots(2, 1, figsize=(10, 9), constrained_layout=True)

pcm0 = axes[0].pcolormesh(
    tof_edges_crop,
    delay_edges_diag,
    rel_map_crop,
    shading="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
fig.colorbar(pcm0, ax=axes[0], label="relative change (%)")
axes[0].set_xlabel("eTOF (100 ps ticks)")
axes[0].set_ylabel("delay (fs)")
axes[0].set_title("Relative-change map in raw TOF coordinates")

ke_edges_plot_diag, rel_map_ke_diag = (
    (ke_edges_crop[::-1], rel_map_crop[:, ::-1])
    if ke_edges_crop[0] > ke_edges_crop[-1]
    else (ke_edges_crop, rel_map_crop)
)
pcm1 = axes[1].pcolormesh(
    ke_edges_plot_diag,
    delay_edges_diag,
    rel_map_ke_diag,
    shading="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
fig.colorbar(pcm1, ax=axes[1], label="relative change (%)")
axes[1].set_xlabel("electron kinetic energy (eV)")
axes[1].set_ylabel("delay (fs)")
axes[1].set_xlim(DIAG_KE_RANGE_EV)
axes[1].set_title("Same relative-change map after TOF -> KE calibration")

fig.suptitle(f"{DIAG_ENERGY_EV:.1f} eV spectral-drift diagnostic", fontsize=13)
plt.show()

# Text Summary Output
print("Diagnostic summary:")
print(
    f"KE centroid range: {np.nanmin(ke_centroid_z):.4f} to {np.nanmax(ke_centroid_z):.4f} eV"
)
print(
    f"KE centroid peak-to-peak: {np.nanmax(ke_centroid_z) - np.nanmin(ke_centroid_z):.4f} eV"
)
print(
    f"TOF centroid peak-to-peak: {np.nanmax(tof_centroid_z) - np.nanmin(tof_centroid_z):.4f}"
)
for ig, label in enumerate(gmd_labels):
    frac = 100.0 * gmd_shot_fraction[ig]
    print(f"GMD bin {label}: {np.nanmin(frac):.2f}% to {np.nanmax(frac):.2f}% of shots")


### GMD-bin-resolved relative-change maps

In [ ]:
# Choose one exact loaded_aggs key. Run this once for 150C_274.5 eV and once for 7June_147C_274.5 eV.
GMD_DIAG_RUN_KEY = "glycine_delay_scan_7June_147C_274.5eV_aggregates_tr_etof1.h5"
DIAG_KE_RANGE_EV = (220.0, 270.0)

# Load selected aggregate
if GMD_DIAG_RUN_KEY not in loaded_aggs:
    raise KeyError(
        f"{GMD_DIAG_RUN_KEY!r} not found in loaded_aggs.\nAvailable keys include:\n"
        + "\n".join(sorted(loaded_aggs.keys()))
    )

item = loaded_aggs[GMD_DIAG_RUN_KEY]
a = item["agg"]

D = np.asarray(a.D, dtype=float)  # (gmd, delay, tof)
G = np.asarray(a.G, dtype=float)  # (gmd, delay)
n = np.asarray(a.n_per_bin, dtype=float)  # (gmd, delay)

tof_edges = np.asarray(a.tof_edges, dtype=float)
gmd_edges = np.asarray(a.gmd_edges, dtype=float)
z_edges = np.asarray(a.z_edges, dtype=float)

z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
delay_edges_fs = (z_edges - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT
delay_cent_fs = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

if D.ndim != 3:
    raise ValueError(f"Expected D[gmd, delay, tof], got {D.shape}")
if D.shape[:2] != n.shape:
    raise ValueError(
        f"D leading dimensions {D.shape[:2]} do not match n_per_bin shape {n.shape}"
    )
if G.shape != n.shape:
    raise ValueError(f"G shape {G.shape} does not match n_per_bin shape {n.shape}")

# KE axis and valid diagnostic interval
ke_edges = tof_to_ke(tof_edges)
ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])
valid_ke_bins = (
    np.isfinite(ke_edges[:-1]) & np.isfinite(ke_edges[1:]) & np.isfinite(ke_cent)
)
diag_mask = (
    valid_ke_bins & (ke_cent >= DIAG_KE_RANGE_EV[0]) & (ke_cent <= DIAG_KE_RANGE_EV[1])
)

if not np.any(diag_mask):
    raise ValueError(f"No valid KE bins inside {DIAG_KE_RANGE_EV} eV.")

diag_idx = np.where(diag_mask)[0]
lo_i, hi_i = int(diag_idx[0]), int(diag_idx[-1]) + 1
ke_edges_crop = ke_edges[lo_i : hi_i + 1]

# Reconstruct raw counts separately for every GMD bin
counts_g_z_tof = D * n[..., None]

# Relative-change maps per GMD bin
rel_maps_g = []
for ig in range(D.shape[0]):
    counts_z_tof = counts_g_z_tof[ig]
    total_electrons_z = np.nansum(counts_z_tof, axis=1)

    with np.errstate(invalid="ignore", divide="ignore"):
        spectrum_fraction = counts_z_tof / total_electrons_z[:, None]

    avg_spectrum = np.nanmean(spectrum_fraction, axis=0)

    with np.errstate(invalid="ignore", divide="ignore"):
        rel_map = (
            100.0 * (spectrum_fraction - avg_spectrum[None, :]) / avg_spectrum[None, :]
        )

    rel_maps_g.append(rel_map[:, lo_i:hi_i])

# Calculate the ordinary map with all GMD bins combined
counts_z_tof_all = np.nansum(counts_g_z_tof, axis=0)
total_electrons_z_all = np.nansum(counts_z_tof_all, axis=1)

with np.errstate(invalid="ignore", divide="ignore"):
    spectrum_fraction_all = counts_z_tof_all / total_electrons_z_all[:, None]

avg_spectrum_all = np.nanmean(spectrum_fraction_all, axis=0)

with np.errstate(invalid="ignore", divide="ignore"):
    rel_map_all = (
        100.0
        * (spectrum_fraction_all - avg_spectrum_all[None, :])
        / avg_spectrum_all[None, :]
    )

rel_map_all = rel_map_all[:, lo_i:hi_i]

# Put KE in increasing order for plotting
if ke_edges_crop[0] > ke_edges_crop[-1]:
    ke_edges_plot = ke_edges_crop[::-1]
    rel_maps_g = [m[:, ::-1] for m in rel_maps_g]
    rel_map_all = rel_map_all[:, ::-1]
else:
    ke_edges_plot = ke_edges_crop

# Use one common colour scale for every panel
all_finite = [
    m[np.isfinite(m)] for m in rel_maps_g + [rel_map_all] if m[np.isfinite(m)].size
]
vmax = np.nanpercentile(np.concatenate(all_finite), 98) if all_finite else 10.0
if not np.isfinite(vmax) or vmax <= 0:
    vmax = 10.0

# Plot four GMD bins + combined map
n_gmd = D.shape[0]
fig, axes = plt.subplots(
    1, n_gmd + 1, figsize=(3.4 * (n_gmd + 1), 4.8), sharey=True, constrained_layout=True
)

for ig in range(n_gmd):
    pcm = axes[ig].pcolormesh(
        ke_edges_plot,
        delay_edges_fs,
        rel_maps_g[ig],
        shading="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    axes[ig].set_title(f"GMD {gmd_edges[ig]:.0f}-{gmd_edges[ig + 1]:.0f}")
    axes[ig].set_xlabel("electron kinetic energy (eV)")
    axes[ig].set_xlim(DIAG_KE_RANGE_EV)

axes[-1].pcolormesh(
    ke_edges_plot,
    delay_edges_fs,
    rel_map_all,
    shading="auto",
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
)
axes[-1].set_title("all GMD bins")
axes[-1].set_xlabel("electron kinetic energy (eV)")
axes[-1].set_xlim(DIAG_KE_RANGE_EV)

axes[0].set_ylabel("delay (fs)")
fig.colorbar(pcm, ax=axes, label="relative change (%)")
fig.suptitle(
    f"{item['energy']:.1f} eV - {GMD_DIAG_RUN_KEY}\nGMD-bin-resolved electron-yield maps",
    fontsize=12,
)
plt.show()

# Print GMD-bin statistics
print("Run:", GMD_DIAG_RUN_KEY)
print("Energy:", item["energy"], "eV\n")
for ig in range(n_gmd):
    print(
        f"GMD {gmd_edges[ig]:.0f}-{gmd_edges[ig + 1]:.0f}: {int(np.nansum(n[ig]))} shots total"
    )


### Individual aggregate runs at one photon energy

In [ ]:
# Choose nominal photon energy. 274.5 eV is useful because you have both 150C and 147C.
COMPARE_ENERGY_EV = 274.0
LOW_WINDOW_KE_EV = (225.0, 240.0)
HIGH_WINDOW_KE_EV = (245.0, 266.0)
DIAG_KE_RANGE_EV = (220.0, 270.0)


# Helpers
def _run_compare_ke_weights(tof_edges, ke_min, ke_max):
    ke_edges = tof_to_ke(np.asarray(tof_edges, dtype=float))
    e0, e1 = ke_edges[:-1], ke_edges[1:]
    valid = np.isfinite(e0) & np.isfinite(e1)
    bin_lo, bin_hi = np.minimum(e0, e1), np.maximum(e0, e1)
    bin_width = bin_hi - bin_lo
    overlap = np.maximum(0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min))

    weights = np.zeros_like(bin_width, dtype=float)
    good = valid & (bin_width > 0)
    weights[good] = overlap[good] / bin_width[good]
    return weights


def _run_compare_relative(y):
    y = np.asarray(y, dtype=float)
    mean = np.nanmean(y)
    with np.errstate(invalid="ignore", divide="ignore"):
        return 100.0 * (y - mean) / mean


def _short_run_name(key):
    name = Path(key).stem
    name = name.replace("glycine_delay_scan_", "")
    name = name.replace("_aggregates_tr_etof1", "")
    return name


# Select files WITHOUT merging them
matching = [
    (key, item)
    for key, item in loaded_aggs.items()
    if np.isclose(item["energy"], COMPARE_ENERGY_EV)
]
matching = sorted(matching, key=lambda x: x[0])

if len(matching) < 2:
    raise ValueError(
        f"Only {len(matching)} file(s) found at {COMPARE_ENERGY_EV:.1f} eV."
    )

print(f"Comparing {len(matching)} separate files at {COMPARE_ENERGY_EV:.1f} eV:")
for key, _ in matching:
    print("  ", key)

# Process each run independently
results = {}

for key, item in matching:
    a = item["agg"]
    D, G, n = (
        np.asarray(a.D, dtype=float),
        np.asarray(a.G, dtype=float),
        np.asarray(a.n_per_bin, dtype=float),
    )
    tof_edges = np.asarray(a.tof_edges, dtype=float)
    gmd_edges = np.asarray(a.gmd_edges, dtype=float)
    z_edges = np.asarray(a.z_edges, dtype=float)

    z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
    delay_edges_fs = (z_edges - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT
    delay_cent_fs = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

    # Raw reconstructed counts
    counts_g_z_tof = D * n[..., None]
    counts_z_tof = np.nansum(counts_g_z_tof, axis=0)
    total_electrons_z = np.nansum(counts_z_tof, axis=1)
    shots_z = np.nansum(n, axis=0)
    gmdsum_z = np.nansum(G * n, axis=0)

    # Electron-yield-normalised relative-change map
    with np.errstate(invalid="ignore", divide="ignore"):
        spectrum_fraction = counts_z_tof / total_electrons_z[:, None]

    avg_spectrum = np.nanmean(spectrum_fraction, axis=0)

    with np.errstate(invalid="ignore", divide="ignore"):
        rel_map = (
            100.0 * (spectrum_fraction - avg_spectrum[None, :]) / avg_spectrum[None, :]
        )

    # KE axis
    ke_edges = tof_to_ke(tof_edges)
    ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])
    valid_ke = (
        np.isfinite(ke_edges[:-1]) & np.isfinite(ke_edges[1:]) & np.isfinite(ke_cent)
    )
    diag_mask = (
        valid_ke & (ke_cent >= DIAG_KE_RANGE_EV[0]) & (ke_cent <= DIAG_KE_RANGE_EV[1])
    )

    idx = np.where(diag_mask)[0]
    if idx.size == 0:
        raise ValueError(f"{key}: no KE bins inside {DIAG_KE_RANGE_EV} eV.")

    lo_i, hi_i = int(idx[0]), int(idx[-1]) + 1
    map_crop = rel_map[:, lo_i:hi_i]
    ke_edges_crop = ke_edges[lo_i : hi_i + 1]

    if ke_edges_crop[0] > ke_edges_crop[-1]:
        ke_edges_plot = ke_edges_crop[::-1]
        map_crop = map_crop[:, ::-1]
    else:
        ke_edges_plot = ke_edges_crop

    # Broad thesis windows + complementary spectral regions
    w_low = _run_compare_ke_weights(tof_edges, LOW_WINDOW_KE_EV[0], LOW_WINDOW_KE_EV[1])
    w_high = _run_compare_ke_weights(
        tof_edges, HIGH_WINDOW_KE_EV[0], HIGH_WINDOW_KE_EV[1]
    )
    w_diag = _run_compare_ke_weights(
        tof_edges, DIAG_KE_RANGE_EV[0], DIAG_KE_RANGE_EV[1]
    )
    w_other = np.clip(w_diag - w_low - w_high, 0.0, 1.0)

    C_low = np.nansum(counts_z_tof * w_low[None, :], axis=1)
    C_high = np.nansum(counts_z_tof * w_high[None, :], axis=1)
    C_other = np.nansum(counts_z_tof * w_other[None, :], axis=1)
    C_diag = np.nansum(counts_z_tof * w_diag[None, :], axis=1)
    C_outside = total_electrons_z - C_diag

    with np.errstate(invalid="ignore", divide="ignore"):
        Y_low = C_low / total_electrons_z
        Y_high = C_high / total_electrons_z
        Y_other = C_other / total_electrons_z
        Y_outside = C_outside / total_electrons_z

    # GMD-bin populations
    with np.errstate(invalid="ignore", divide="ignore"):
        gmd_shot_fraction = n / np.nansum(n, axis=0)[None, :]

    results[key] = {
        "delay_edges_fs": delay_edges_fs,
        "delay_cent_fs": delay_cent_fs,
        "ke_edges_plot": ke_edges_plot,
        "map_crop": map_crop,
        "low_rel": _run_compare_relative(Y_low),
        "high_rel": _run_compare_relative(Y_high),
        "other_rel": _run_compare_relative(Y_other),
        "outside_rel": _run_compare_relative(Y_outside),
        "gmd_shot_fraction": gmd_shot_fraction,
        "gmd_edges": gmd_edges,
        "total_electrons": total_electrons_z,
        "gmdsum": gmdsum_z,
        "shots": shots_z,
    }

# Common colour scale across all run maps
finite_arrays = [
    r["map_crop"][np.isfinite(r["map_crop"])]
    for r in results.values()
    if r["map_crop"][np.isfinite(r["map_crop"])].size
]
vmax = 40 if finite_arrays else 10.0

# Plot: row 1 (2D map), row 2 (yield traces), row 3 (GMD bins)
n_runs = len(matching)
fig, axes = plt.subplots(
    3, n_runs, figsize=(5.0 * n_runs, 10), squeeze=False, constrained_layout=True
)

for col, (key, item) in enumerate(matching):
    r = results[key]
    t = r["delay_cent_fs"]

    # 1. Relative-change map
    pcm = axes[0, col].pcolormesh(
        r["ke_edges_plot"],
        r["delay_edges_fs"],
        r["map_crop"],
        shading="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    axes[0, col].set_xlim(DIAG_KE_RANGE_EV)
    axes[0, col].set_xlabel("electron kinetic energy (eV)")
    axes[0, col].set_title(_short_run_name(key))

    # 2. Spectral-region traces
    axes[1, col].plot(
        t,
        r["low_rel"],
        "o-",
        ms=3,
        lw=1.0,
        label=f"low {LOW_WINDOW_KE_EV[0]:.0f}-{LOW_WINDOW_KE_EV[1]:.0f}",
    )
    axes[1, col].plot(
        t,
        r["high_rel"],
        "o-",
        ms=3,
        lw=1.0,
        label=f"high {HIGH_WINDOW_KE_EV[0]:.0f}-{HIGH_WINDOW_KE_EV[1]:.0f}",
    )
    axes[1, col].plot(t, r["other_rel"], "o-", ms=3, lw=1.0, label="other 220-270")
    axes[1, col].plot(t, r["outside_rel"], "o-", ms=3, lw=1.0, label="outside 220-270")
    axes[1, col].axhline(0.0, color="k", lw=0.7, alpha=0.4)
    axes[1, col].set_xlabel("delay (fs)")
    axes[1, col].grid(alpha=0.2)

    # 3. GMD-bin population
    for ig in range(r["gmd_shot_fraction"].shape[0]):
        axes[2, col].plot(
            t,
            100.0 * r["gmd_shot_fraction"][ig],
            "o-",
            ms=3,
            lw=1.0,
            label=f"{r['gmd_edges'][ig]:.0f}-{r['gmd_edges'][ig + 1]:.0f}",
        )
    axes[2, col].set_xlabel("delay (fs)")
    axes[2, col].grid(alpha=0.2)

# Global labels & styling
axes[0, 0].set_ylabel("delay (fs)")
axes[1, 0].set_ylabel("relative electron yield (%)")
axes[2, 0].set_ylabel("shots in GMD bin (%)")
axes[1, 0].legend(fontsize=8)
axes[2, 0].legend(title="GMD bin", fontsize=8)

fig.colorbar(pcm, ax=axes[0, :], label="relative change (%)")
fig.suptitle(
    f"Individual-run comparison at {COMPARE_ENERGY_EV:.1f} eV\nNo merging, no residual-gas subtraction, no detrending",
    fontsize=13,
)

plt.show()


### Repeated scans reconstructed from acquisition order

In [ ]:
# Uses the NON-AGGREGATE combined H5:
#      z       : shot-wise delay/stage coordinate, shape (train, pulse)
#      tofs_e  : eTOF peak times, shape (train, pulse, max_peaks)
# The aggregate z_edges and tof_edges are reused, so the delay and TOF
# binning is the same as in the aggregate analysis.
# Repeated scans are reconstructed from acquisition order.
# No residual-gas subtraction. No smoothing. No interpolation.
# No detrending. No Jacobian.

# User choice.
SCAN_NAME = "glycine_delay_scan_150C_274.5eV"
COMBINED_DIR = Path(config.COMBINED_DIR)
RAW_H5 = COMBINED_DIR / f"{SCAN_NAME}.h5"
AGG_KEY = f"{SCAN_NAME}_aggregates_tr_etof1.h5"

LOW_WINDOW_KE_EV = (225.0, 240.0)
HIGH_WINDOW_KE_EV = (245.0, 266.0)
DIAG_KE_RANGE_EV = (220.0, 270.0)

# A genuine new scan should contain a reasonable fraction of the
# full delay range. This removes isolated acquisition fragments.
MIN_UNIQUE_DELAY_BINS = 10
# A reset must jump backwards by at least this many normal delay steps.
RESET_MIN_STEPS = 10.0

# Check inputs.
if not RAW_H5.exists():
    raise FileNotFoundError(f"Combined H5 not found:\n{RAW_H5}")

if AGG_KEY not in loaded_aggs:
    raise KeyError(f"{AGG_KEY!r} is not present in loaded_aggs.")

agg_scan = loaded_aggs[AGG_KEY]["agg"]
z_edges = np.asarray(agg_scan.z_edges, dtype=float)
tof_edges = np.asarray(agg_scan.tof_edges, dtype=float)
z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
delay_cent_fs_all = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

# KE-window weights using the existing calibration.
ke_edges = tof_to_ke(tof_edges)
ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])
valid_ke = np.isfinite(ke_edges[:-1]) & np.isfinite(ke_edges[1:]) & np.isfinite(ke_cent)


def ke_window_weights(ke_min, ke_max):
    """Fractional overlap of each calibrated TOF bin with a KE interval."""
    e0, e1 = ke_edges[:-1], ke_edges[1:]
    bin_lo = np.minimum(e0, e1)
    bin_hi = np.maximum(e0, e1)
    width = bin_hi - bin_lo
    overlap = np.maximum(0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min))

    weights = np.zeros_like(width, dtype=float)
    good = valid_ke & (width > 0)
    weights[good] = overlap[good] / width[good]
    return weights


w_low = ke_window_weights(*LOW_WINDOW_KE_EV)
w_high = ke_window_weights(*HIGH_WINDOW_KE_EV)
w_diag = ke_window_weights(*DIAG_KE_RANGE_EV)
# Everything inside 220-270 eV that is not in either broad thesis window.
w_other = np.clip(w_diag - w_low - w_high, 0.0, 1.0)


def relative_change(y):
    """Relative change with respect to the mean over delay."""
    y = np.asarray(y, dtype=float)
    mean = np.nanmean(y)
    with np.errstate(invalid="ignore", divide="ignore"):
        return 100.0 * (y - mean) / mean


# Read the shot-wise delay coordinate.
# z is small enough to load completely. tofs_e is deliberately NOT.
with h5py.File(RAW_H5, "r") as f:
    if "z" not in f:
        raise KeyError("'z' not found in combined H5.")
    if "tofs_e" not in f:
        raise KeyError("'tofs_e' not found in combined H5.")

    z = np.asarray(f["z"][:], dtype=float)
    etof_ds = f["tofs_e"]

    print("z shape:     ", z.shape)
    print("tofs_e shape:", etof_ds.shape)

    if z.shape != etof_ds.shape[:2]:
        raise ValueError(
            f"z shot dimensions do not match tofs_e:\nz = {z.shape}\ntofs_e = {etof_ds.shape}"
        )

# Assign every shot to the SAME delay bins used by the aggregate.
delay_bin_pulse = np.searchsorted(z_edges, z, side="right") - 1
valid_delay_pulse = (
    np.isfinite(z) & (delay_bin_pulse >= 0) & (delay_bin_pulse < len(z_cent))
)

# Determine one representative delay bin per train.
# 101 shots per train. Most common delay bin within each train is used.
n_trains = z.shape[0]
train_delay_bin = np.full(n_trains, -1, dtype=int)
train_delay_purity = np.full(n_trains, np.nan, dtype=float)

for itr in range(n_trains):
    valid_this_train = valid_delay_pulse[itr]
    bins = delay_bin_pulse[itr, valid_this_train]

    if bins.size == 0:
        continue

    values, counts = np.unique(bins, return_counts=True)
    imax = np.argmax(counts)
    train_delay_bin[itr] = values[imax]
    train_delay_purity[itr] = counts[imax] / bins.size

valid_train = train_delay_bin >= 0
if not np.any(valid_train):
    raise ValueError("No valid trains could be assigned to the aggregate delay bins.")

print(f"\nValid trains: {np.sum(valid_train)} / {n_trains}")
print(
    f"Median fraction of shots agreeing with the assigned train delay: {100*np.nanmedian(train_delay_purity[valid_train]):.2f}%"
)
print(
    f"Minimum train-delay agreement: {100*np.nanmin(train_delay_purity[valid_train]):.2f}%"
)
poor_train_fraction = np.mean(train_delay_purity[valid_train] < 0.90)
print(f"Trains with <90% delay-bin agreement: {100*poor_train_fraction:.2f}%")

# Build contiguous delay-point visits in acquisition order.
# One block = consecutive trains recorded at one nominal delay setting.
valid_train_indices = np.where(valid_train)[0]
blocks = []
start_train = valid_train_indices[0]
previous_train = valid_train_indices[0]
current_bin = train_delay_bin[start_train]

for itr in valid_train_indices[1:]:
    same_delay = train_delay_bin[itr] == current_bin
    contiguous = itr == previous_train + 1

    if not same_delay or not contiguous:
        blocks.append(
            {
                "start": start_train,
                "stop": previous_train + 1,
                "delay_bin": int(current_bin),
            }
        )
        start_train = itr
        current_bin = train_delay_bin[itr]
    previous_train = itr

blocks.append(
    {
        "start": start_train,
        "stop": previous_train + 1,
        "delay_bin": int(current_bin),
    }
)

block_bins = np.array([b["delay_bin"] for b in blocks], dtype=int)
block_delay_fs = delay_cent_fs_all[block_bins]
print(f"\nDelay-point visits found: {len(blocks)}")

# Determine the repeated scans.
# Small reversals remain part of the same scan. Only large jumps reset.
ddelay = np.diff(block_delay_fs)
nonzero_steps = np.abs(ddelay[np.isfinite(ddelay) & (np.abs(ddelay) > 0)])
if nonzero_steps.size == 0:
    raise ValueError("Could not determine a delay step from the acquisition sequence.")
typical_step = np.nanmedian(nonzero_steps)

# Determine normal scan direction using ordinary steps, excluding resets.
ordinary_steps = ddelay[
    np.isfinite(ddelay) & (np.abs(ddelay) > 0) & (np.abs(ddelay) <= 2.0 * typical_step)
]
if ordinary_steps.size == 0:
    raise ValueError("Could not determine the normal delay-scan direction.")

n_positive_steps = np.sum(ordinary_steps > 0)
n_negative_steps = np.sum(ordinary_steps < 0)
if n_positive_steps > n_negative_steps:
    dominant_direction = 1.0
elif n_negative_steps > n_positive_steps:
    dominant_direction = -1.0
else:
    raise ValueError("Normal delay-scan direction is ambiguous.")

scan_starts = [0]
for i, step in enumerate(ddelay):
    if not np.isfinite(step):
        continue
    is_large_jump = abs(step) >= RESET_MIN_STEPS * typical_step
    is_opposite_direction = np.sign(step) == -dominant_direction
    if is_large_jump and is_opposite_direction:
        scan_starts.append(i + 1)

scan_starts = sorted(set(scan_starts))
scan_stops = scan_starts[1:] + [len(blocks)]
scan_block_ranges_all = list(zip(scan_starts, scan_stops))

# Reject isolated/incomplete fragments.
scan_block_ranges = []
excluded_scan_fragments = []

for ib0, ib1 in scan_block_ranges_all:
    delay_bins_this_scan = block_bins[ib0:ib1]
    n_unique = len(np.unique(delay_bins_this_scan))
    if n_unique >= MIN_UNIQUE_DELAY_BINS:
        scan_block_ranges.append((ib0, ib1))
    else:
        excluded_scan_fragments.append((ib0, ib1))

direction_text = "increasing delay" if dominant_direction > 0 else "decreasing delay"
print(f"Typical delay step: {typical_step:.4f} fs")
print(f"Dominant scan direction: {direction_text}")
print(f"Detected complete/relevant scans: {len(scan_block_ranges)}\n")

for iscan, (ib0, ib1) in enumerate(scan_block_ranges, start=1):
    delays = block_delay_fs[ib0:ib1]
    delay_bins_this_scan = block_bins[ib0:ib1]
    print(
        f"scan {iscan}: {len(delays)} delay-point visits, {len(np.unique(delay_bins_this_scan))} unique delay bins, {delays[0]:.3f} -> {delays[-1]:.3f} fs"
    )

if excluded_scan_fragments:
    print("\nExcluded incomplete acquisition fragment(s):")
    for ib0, ib1 in excluded_scan_fragments:
        delays = block_delay_fs[ib0:ib1]
        delay_bins_fragment = block_bins[ib0:ib1]
        print(
            f"  {len(delays)} visit(s), {len(np.unique(delay_bins_fragment))} unique bin(s): {delays[0]:.3f} -> {delays[-1]:.3f} fs"
        )

if len(scan_block_ranges) == 0:
    raise ValueError("No complete delay scans survived the scan-selection criteria.")

# Reconstruct the eTOF spectrum independently for every repeated scan.
# Read chunk-by-chunk rather than loading everything into RAM.
scan_results = []
with h5py.File(RAW_H5, "r") as f:
    etof_ds = f["tofs_e"]

    for iscan, (ib0, ib1) in enumerate(scan_block_ranges, start=1):
        spectra_by_delay = {}
        shots_by_delay = {}

        for block in blocks[ib0:ib1]:
            delay_bin = block["delay_bin"]
            start, stop = block["start"], block["stop"]

            # Read only the trains belonging to this delay-point visit
            etof_chunk = np.asarray(etof_ds[start:stop])
            pulse_delay_bins = delay_bin_pulse[start:stop]
            valid_shots = valid_delay_pulse[start:stop] & (
                pulse_delay_bins == delay_bin
            )
            n_valid_shots = int(np.sum(valid_shots))
            if n_valid_shots == 0:
                continue

            # Select electron peak times from matching shots
            events = etof_chunk[valid_shots].ravel()
            events = events[events > 0]  # drop zero padding
            events = events[(events >= tof_edges[0]) & (events < tof_edges[-1])]

            hist, _ = np.histogram(events, bins=tof_edges)

            if delay_bin not in spectra_by_delay:
                spectra_by_delay[delay_bin] = np.zeros(len(tof_edges) - 1, dtype=float)
                shots_by_delay[delay_bin] = 0

            spectra_by_delay[delay_bin] += hist
            shots_by_delay[delay_bin] += n_valid_shots

        if len(spectra_by_delay) < 2:
            print(f"Skipping scan {iscan}: fewer than two reconstructed delay bins.")
            continue

        # Sort by physical delay for plotting and fitting
        delay_bins_scan = np.array(sorted(spectra_by_delay), dtype=int)
        counts_z_tof = np.stack([spectra_by_delay[i] for i in delay_bins_scan], axis=0)
        delay_fs = delay_cent_fs_all[delay_bins_scan]
        shots_z = np.array([shots_by_delay[i] for i in delay_bins_scan], dtype=float)
        total_electrons_z = np.nansum(counts_z_tof, axis=1)

        # Electron-yield-normalised spectrum
        with np.errstate(invalid="ignore", divide="ignore"):
            spectrum_fraction = counts_z_tof / total_electrons_z[:, None]

        avg_spectrum = np.nanmean(spectrum_fraction, axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            rel_map = (
                100.0
                * (spectrum_fraction - avg_spectrum[None, :])
                / avg_spectrum[None, :]
            )

        # Broad KE-region yields
        C_low = np.nansum(counts_z_tof * w_low[None, :], axis=1)
        C_high = np.nansum(counts_z_tof * w_high[None, :], axis=1)
        C_other = np.nansum(counts_z_tof * w_other[None, :], axis=1)
        C_diag = np.nansum(counts_z_tof * w_diag[None, :], axis=1)
        C_outside = total_electrons_z - C_diag

        with np.errstate(invalid="ignore", divide="ignore"):
            Y_low = C_low / total_electrons_z
            Y_high = C_high / total_electrons_z
            Y_other = C_other / total_electrons_z
            Y_outside = C_outside / total_electrons_z

        low_rel = relative_change(Y_low)
        high_rel = relative_change(Y_high)
        other_rel = relative_change(Y_other)
        outside_rel = relative_change(Y_outside)

        # Linear diagnostic fits (nothing is subtracted from data)
        good_low = np.isfinite(delay_fs) & np.isfinite(low_rel)
        good_high = np.isfinite(delay_fs) & np.isfinite(high_rel)
        good_other = np.isfinite(delay_fs) & np.isfinite(other_rel)
        good_outside = np.isfinite(delay_fs) & np.isfinite(outside_rel)

        fit_low = linregress(delay_fs[good_low], low_rel[good_low])
        fit_high = linregress(delay_fs[good_high], high_rel[good_high])
        fit_other = linregress(delay_fs[good_other], other_rel[good_other])
        fit_outside = linregress(delay_fs[good_outside], outside_rel[good_outside])

        scan_results.append(
            {
                "scan": iscan,
                "delay_fs": delay_fs,
                "rel_map": rel_map,
                "low_rel": low_rel,
                "high_rel": high_rel,
                "other_rel": other_rel,
                "outside_rel": outside_rel,
                "shots": shots_z,
                "total_electrons": total_electrons_z,
                "low_fit": fit_low,
                "high_fit": fit_high,
                "other_fit": fit_other,
                "outside_fit": fit_outside,
            }
        )

if len(scan_results) == 0:
    raise ValueError("No repeated scan spectra could be reconstructed.")

# Crop 2D maps to the diagnostic kinetic-energy range.
diag_mask = (
    valid_ke & (ke_cent >= DIAG_KE_RANGE_EV[0]) & (ke_cent <= DIAG_KE_RANGE_EV[1])
)
idx = np.where(diag_mask)[0]
if idx.size == 0:
    raise ValueError(f"No calibrated KE bins inside {DIAG_KE_RANGE_EV} eV.")
lo_i, hi_i = int(idx[0]), int(idx[-1]) + 1

ke_edges_crop = ke_edges[lo_i : hi_i + 1]
maps = [r["rel_map"][:, lo_i:hi_i] for r in scan_results]

# Put KE in increasing order for plotting
if ke_edges_crop[0] > ke_edges_crop[-1]:
    ke_edges_plot = ke_edges_crop[::-1]
    maps = [m[:, ::-1] for m in maps]
else:
    ke_edges_plot = ke_edges_crop

# Shared colour scale across all scans
finite_maps = [m[np.isfinite(m)] for m in maps if np.any(np.isfinite(m))]
if finite_maps:
    vmax = np.nanpercentile(np.abs(np.concatenate(finite_maps)), 98)
else:
    vmax = 10.0
if not np.isfinite(vmax) or vmax <= 0:
    vmax = 10.0

# Plot one column per repeated scan.
n_scans = len(scan_results)
fig, axes = plt.subplots(
    2, n_scans, figsize=(4.2 * n_scans, 8), squeeze=False, constrained_layout=True
)

for col, result in enumerate(scan_results):
    t = result["delay_fs"]
    if len(t) < 2:
        continue

    # Delay-bin plotting edges
    t_edges = np.empty(len(t) + 1, dtype=float)
    t_edges[1:-1] = 0.5 * (t[:-1] + t[1:])
    t_edges[0] = t[0] - 0.5 * (t[1] - t[0])
    t_edges[-1] = t[-1] + 0.5 * (t[-1] - t[-2])

    # Top: 2D map
    pcm = axes[0, col].pcolormesh(
        ke_edges_plot,
        t_edges,
        maps[col],
        shading="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    axes[0, col].set_xlim(DIAG_KE_RANGE_EV)
    axes[0, col].set_xlabel("electron kinetic energy (eV)")
    axes[0, col].set_title(f"scan {result['scan']}")

    # Bottom: Broad-region traces
    axes[1, col].plot(t, result["low_rel"], "o-", ms=3, lw=1.0, label="low 225-240")
    axes[1, col].plot(t, result["high_rel"], "o-", ms=3, lw=1.0, label="high 245-266")
    axes[1, col].plot(t, result["other_rel"], "o-", ms=3, lw=1.0, label="other 220-270")
    axes[1, col].plot(
        t, result["outside_rel"], "o-", ms=3, lw=1.0, label="outside 220-270"
    )

    # Dashed lines: Diagnostic linear fits (LOW and HIGH only)
    low_fit, high_fit = result["low_fit"], result["high_fit"]
    axes[1, col].plot(t, low_fit.intercept + low_fit.slope * t, "--", lw=1.0, alpha=0.7)
    axes[1, col].plot(
        t, high_fit.intercept + high_fit.slope * t, "--", lw=1.0, alpha=0.7
    )

    axes[1, col].axhline(0.0, color="k", lw=0.7, alpha=0.4)
    axes[1, col].set_xlabel("delay (fs)")
    axes[1, col].grid(alpha=0.2)

axes[0, 0].set_ylabel("delay (fs)")
axes[1, 0].set_ylabel("relative electron yield (%)")
axes[1, 0].legend(fontsize=8)

fig.colorbar(pcm, ax=axes[0, :], label="relative change (%)")
fig.suptitle(
    f"{SCAN_NAME}\nIndividual repeated delay scans - no averaging between scans",
    fontsize=13,
)
plt.show()

# Numerical slope comparison.
print(
    "\nLinear-slope diagnostic\n(fits quantify the gradient only; nothing has been detrended)\n"
)
for result in scan_results:
    low, high = result["low_fit"], result["high_fit"]
    other, outside = result["other_fit"], result["outside_fit"]

    print(f"scan {result['scan']}:")
    print(
        f"  low 225-240 eV:     {low.slope:+.4f} %/fs   r={low.rvalue:+.3f}, p={low.pvalue:.3g}"
    )
    print(
        f"  high 245-266 eV:    {high.slope:+.4f} %/fs   r={high.rvalue:+.3f}, p={high.pvalue:.3g}"
    )
    print(
        f"  other 220-270 eV:   {other.slope:+.4f} %/fs   r={other.rvalue:+.3f}, p={other.pvalue:.3g}"
    )
    print(
        f"  outside 220-270 eV: {outside.slope:+.4f} %/fs   r={outside.rvalue:+.3f}, p={outside.pvalue:.3g}\n"
    )


### Gradient correlations with acquisition variables

In [ ]:
# For one selected run this cell:
#   1. reconstructs the individual repeated delay scans from acquisition order;
#   2. reconstructs the eTOF spectrum independently for each scan;
#   3. tracks the spectral redistribution ("outside 220-270 eV");
#   4. tracks:
#          - electrons / shot
#          - mean GMD / shot
#          - hor_pos
#          - ver_pos
#          - mpe
#          - z_std
#   5. plots all diagnostics versus:
#          A) physical pump-probe delay
#          B) actual acquisition order (global train index)
#   6. prints correlations between the spectral redistribution and the
#      experimental diagnostics within each repeated scan.
# IMPORTANT:
#   - No residual-gas subtraction.
#   - No smoothing.
#   - No interpolation.
#   - No detrending.
#   - No Jacobian.
# The raw variable names hor_pos, ver_pos and mpe are kept deliberately.
# Their physical interpretation is not assumed here.

# User choice.
SCAN_NAME = "glycine_delay_scan_150C_274.5eV"
COMBINED_DIR = Path(config.COMBINED_DIR)
RAW_H5 = COMBINED_DIR / f"{SCAN_NAME}.h5"
AGG_KEY = f"{SCAN_NAME}_aggregates_tr_etof1.h5"

LOW_WINDOW_KE_EV = (225.0, 240.0)
HIGH_WINDOW_KE_EV = (245.0, 266.0)
DIAG_KE_RANGE_EV = (220.0, 270.0)

MIN_UNIQUE_DELAY_BINS = 10
RESET_MIN_STEPS = 10.0

# Check inputs.
if not RAW_H5.exists():
    raise FileNotFoundError(f"Combined H5 not found:\n{RAW_H5}")

if AGG_KEY not in loaded_aggs:
    raise KeyError(f"{AGG_KEY!r} is not present in loaded_aggs.")

agg_scan = loaded_aggs[AGG_KEY]["agg"]
z_edges = np.asarray(agg_scan.z_edges, dtype=float)
tof_edges = np.asarray(agg_scan.tof_edges, dtype=float)
z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
delay_cent_fs_all = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

# KE-window weights.
ke_edges = tof_to_ke(tof_edges)
ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])
valid_ke = np.isfinite(ke_edges[:-1]) & np.isfinite(ke_edges[1:]) & np.isfinite(ke_cent)


def ke_window_weights(ke_min, ke_max):
    e0 = ke_edges[:-1]
    e1 = ke_edges[1:]
    bin_lo = np.minimum(e0, e1)
    bin_hi = np.maximum(e0, e1)
    width = bin_hi - bin_lo
    overlap = np.maximum(0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min))

    weights = np.zeros_like(width, dtype=float)
    good = valid_ke & (width > 0)
    weights[good] = overlap[good] / width[good]
    return weights


w_low = ke_window_weights(*LOW_WINDOW_KE_EV)
w_high = ke_window_weights(*HIGH_WINDOW_KE_EV)
w_diag = ke_window_weights(*DIAG_KE_RANGE_EV)
w_other = np.clip(w_diag - w_low - w_high, 0.0, 1.0)


def relative_change(y):
    y = np.asarray(y, dtype=float)
    mean = np.nanmean(y)
    with np.errstate(invalid="ignore", divide="ignore"):
        return 100.0 * (y - mean) / mean


def weighted_mean(values, weights):
    """Weighted mean ignoring non-finite values."""
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    good = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(good):
        return np.nan
    return np.sum(values[good] * weights[good]) / np.sum(weights[good])


# Read the small raw diagnostic arrays.
# tofs_e remains on disk and is read only in chunks later.
with h5py.File(RAW_H5, "r") as f:
    required = ["z", "tofs_e", "gmd", "z_std", "hor_pos", "ver_pos", "mpe", "tID"]
    for key in required:
        if key not in f:
            raise KeyError(f"{key!r} not found in {RAW_H5.name}.")

    z = np.asarray(f["z"][:], dtype=float)
    gmd_raw = np.asarray(f["gmd"][:], dtype=float)
    z_std_raw = np.asarray(f["z_std"][:], dtype=float)
    hor_pos_raw = np.asarray(f["hor_pos"][:], dtype=float)
    ver_pos_raw = np.asarray(f["ver_pos"][:], dtype=float)
    mpe_raw = np.asarray(f["mpe"][:], dtype=float)
    tid_raw = np.asarray(f["tID"][:], dtype=float)
    etof_shape = f["tofs_e"].shape

print("z shape:     ", z.shape)
print("tofs_e shape:", etof_shape)

if z.shape != etof_shape[:2]:
    raise ValueError("z shot dimensions do not match tofs_e.")
if gmd_raw.shape != z.shape:
    raise ValueError(f"gmd shape {gmd_raw.shape} does not match z {z.shape}.")
if z_std_raw.shape != z.shape:
    raise ValueError(f"z_std shape {z_std_raw.shape} does not match z {z.shape}.")

n_trains = z.shape[0]

for name, arr in [
    ("hor_pos", hor_pos_raw),
    ("ver_pos", ver_pos_raw),
    ("mpe", mpe_raw),
    ("tID", tid_raw),
]:
    if arr.shape != (n_trains,):
        raise ValueError(f"{name} shape {arr.shape} is not ({n_trains},).")

# Assign every shot to the aggregate delay bins.
delay_bin_pulse = np.searchsorted(z_edges, z, side="right") - 1
valid_delay_pulse = (
    np.isfinite(z) & (delay_bin_pulse >= 0) & (delay_bin_pulse < len(z_cent))
)

# Representative delay bin for each train.
train_delay_bin = np.full(n_trains, -1, dtype=int)
train_delay_purity = np.full(n_trains, np.nan, dtype=float)

for itr in range(n_trains):
    bins = delay_bin_pulse[itr, valid_delay_pulse[itr]]
    if bins.size == 0:
        continue
    values, counts = np.unique(bins, return_counts=True)
    imax = np.argmax(counts)
    train_delay_bin[itr] = values[imax]
    train_delay_purity[itr] = counts[imax] / bins.size

valid_train = train_delay_bin >= 0
print()
print("Valid trains:", np.sum(valid_train), "/", n_trains)
print(
    "Median train-delay agreement:",
    f"{100*np.nanmedian(train_delay_purity[valid_train]):.2f}%",
)

# Build contiguous delay-point visits in acquisition order.
valid_train_indices = np.where(valid_train)[0]
blocks = []
start_train = valid_train_indices[0]
previous_train = valid_train_indices[0]
current_bin = train_delay_bin[start_train]

for itr in valid_train_indices[1:]:
    same_delay = train_delay_bin[itr] == current_bin
    contiguous = itr == previous_train + 1

    if not same_delay or not contiguous:
        blocks.append(
            {
                "start": start_train,
                "stop": previous_train + 1,
                "delay_bin": int(current_bin),
            }
        )
        start_train = itr
        current_bin = train_delay_bin[itr]
    previous_train = itr

blocks.append(
    {
        "start": start_train,
        "stop": previous_train + 1,
        "delay_bin": int(current_bin),
    }
)

block_bins = np.array([b["delay_bin"] for b in blocks], dtype=int)
block_delay_fs = delay_cent_fs_all[block_bins]
print("Delay-point visits found:", len(blocks))

# Find repeated scans from large backward reset jumps.
ddelay = np.diff(block_delay_fs)
nonzero_steps = np.abs(ddelay[np.isfinite(ddelay) & (np.abs(ddelay) > 0)])
typical_step = np.nanmedian(nonzero_steps)
ordinary_steps = ddelay[
    np.isfinite(ddelay) & (np.abs(ddelay) > 0) & (np.abs(ddelay) <= 2.0 * typical_step)
]

n_positive_steps = np.sum(ordinary_steps > 0)
n_negative_steps = np.sum(ordinary_steps < 0)

if n_positive_steps > n_negative_steps:
    dominant_direction = 1.0
elif n_negative_steps > n_positive_steps:
    dominant_direction = -1.0
else:
    raise ValueError("Normal scan direction is ambiguous.")

scan_starts = [0]
for i, step in enumerate(ddelay):
    if not np.isfinite(step):
        continue
    is_large_jump = abs(step) >= RESET_MIN_STEPS * typical_step
    is_opposite_direction = np.sign(step) == -dominant_direction
    if is_large_jump and is_opposite_direction:
        scan_starts.append(i + 1)

scan_starts = sorted(set(scan_starts))
scan_stops = scan_starts[1:] + [len(blocks)]
scan_block_ranges_all = list(zip(scan_starts, scan_stops))

# Reject isolated fragments.
scan_block_ranges = []
for ib0, ib1 in scan_block_ranges_all:
    delay_bins_this_scan = block_bins[ib0:ib1]
    if len(np.unique(delay_bins_this_scan)) >= MIN_UNIQUE_DELAY_BINS:
        scan_block_ranges.append((ib0, ib1))

print("Typical delay step:", f"{typical_step:.4f} fs")
print("Detected complete scans:", len(scan_block_ranges))
print()

for iscan, (ib0, ib1) in enumerate(scan_block_ranges, start=1):
    delays = block_delay_fs[ib0:ib1]
    print(
        f"scan {iscan}: {len(delays)} visits, {len(np.unique(block_bins[ib0:ib1]))} unique bins, {delays[0]:.3f} -> {delays[-1]:.3f} fs"
    )

# Reconstruct spectra AND diagnostic variables per delay bin / scan.
scan_results = []

with h5py.File(RAW_H5, "r") as f:
    etof_ds = f["tofs_e"]

    for iscan, (ib0, ib1) in enumerate(scan_block_ranges, start=1):
        spectra_by_delay = {}
        diagnostics_by_delay = {}
        scan_blocks = blocks[ib0:ib1]
        n_blocks_scan = len(scan_blocks)

        for ivisit, block in enumerate(scan_blocks):
            delay_bin = block["delay_bin"]
            start = block["start"]
            stop = block["stop"]

            # Read this acquisition block only.
            etof_chunk = np.asarray(etof_ds[start:stop])
            pulse_delay_bins = delay_bin_pulse[start:stop]
            valid_shots = valid_delay_pulse[start:stop] & (
                pulse_delay_bins == delay_bin
            )

            shot_weights_per_train = np.sum(valid_shots, axis=1).astype(float)
            n_valid_shots = int(np.sum(shot_weights_per_train))
            if n_valid_shots == 0:
                continue

            # Electron events.
            events = etof_chunk[valid_shots].ravel()
            events = events[events > 0]
            events = events[(events >= tof_edges[0]) & (events < tof_edges[-1])]
            hist, _ = np.histogram(events, bins=tof_edges)

            # Initialise this delay bin.
            if delay_bin not in spectra_by_delay:
                spectra_by_delay[delay_bin] = np.zeros(len(tof_edges) - 1, dtype=float)
                diagnostics_by_delay[delay_bin] = {
                    "shots": 0.0,
                    "gmd_sum": 0.0,
                    "gmd_count": 0.0,
                    "zstd_sum": 0.0,
                    "zstd_count": 0.0,
                    "hor_num": 0.0,
                    "hor_den": 0.0,
                    "ver_num": 0.0,
                    "ver_den": 0.0,
                    "mpe_num": 0.0,
                    "mpe_den": 0.0,
                    "tid_num": 0.0,
                    "tid_den": 0.0,
                    "train_num": 0.0,
                    "train_den": 0.0,
                    "progress_num": 0.0,
                    "progress_den": 0.0,
                }

            spectra_by_delay[delay_bin] += hist
            d = diagnostics_by_delay[delay_bin]
            d["shots"] += n_valid_shots

            # Shot-resolved GMD.
            gmd_values = gmd_raw[start:stop][valid_shots]
            d["gmd_sum"] += np.nansum(gmd_values)
            d["gmd_count"] += np.sum(np.isfinite(gmd_values))

            # Shot-resolved z_std.
            zstd_values = z_std_raw[start:stop][valid_shots]
            d["zstd_sum"] += np.nansum(zstd_values)
            d["zstd_count"] += np.sum(np.isfinite(zstd_values))

            # Train-resolved diagnostics.
            # Weight each train by the number of valid shots from that
            # train contributing to this delay bin.
            train_indices = np.arange(start, stop, dtype=float)
            train_variables = {
                "hor": hor_pos_raw[start:stop],
                "ver": ver_pos_raw[start:stop],
                "mpe": mpe_raw[start:stop],
                "tid": tid_raw[start:stop],
                "train": train_indices,
            }

            for name, values in train_variables.items():
                good = np.isfinite(values) & (shot_weights_per_train > 0)
                d[f"{name}_num"] += np.sum(values[good] * shot_weights_per_train[good])
                d[f"{name}_den"] += np.sum(shot_weights_per_train[good])

            # Acquisition position within this repeated scan.
            progress = ivisit / (n_blocks_scan - 1) if n_blocks_scan > 1 else 0.0
            d["progress_num"] += progress * n_valid_shots
            d["progress_den"] += n_valid_shots

        # Convert one reconstructed scan to arrays.
        delay_bins_scan = np.array(sorted(spectra_by_delay), dtype=int)
        if len(delay_bins_scan) < 2:
            continue

        counts_z_tof = np.stack([spectra_by_delay[i] for i in delay_bins_scan], axis=0)
        delay_fs = delay_cent_fs_all[delay_bins_scan]
        total_electrons_z = np.nansum(counts_z_tof, axis=1)
        diag = [diagnostics_by_delay[i] for i in delay_bins_scan]
        shots_z = np.array([d["shots"] for d in diag], dtype=float)

        with np.errstate(invalid="ignore", divide="ignore"):
            electrons_per_shot = total_electrons_z / shots_z
            mean_gmd = np.array(
                [
                    d["gmd_sum"] / d["gmd_count"] if d["gmd_count"] > 0 else np.nan
                    for d in diag
                ],
                dtype=float,
            )
            mean_zstd = np.array(
                [
                    d["zstd_sum"] / d["zstd_count"] if d["zstd_count"] > 0 else np.nan
                    for d in diag
                ],
                dtype=float,
            )
            mean_hor = np.array(
                [
                    d["hor_num"] / d["hor_den"] if d["hor_den"] > 0 else np.nan
                    for d in diag
                ],
                dtype=float,
            )
            mean_ver = np.array(
                [
                    d["ver_num"] / d["ver_den"] if d["ver_den"] > 0 else np.nan
                    for d in diag
                ],
                dtype=float,
            )
            mean_mpe = np.array(
                [
                    d["mpe_num"] / d["mpe_den"] if d["mpe_den"] > 0 else np.nan
                    for d in diag
                ],
                dtype=float,
            )
            mean_tid = np.array(
                [
                    d["tid_num"] / d["tid_den"] if d["tid_den"] > 0 else np.nan
                    for d in diag
                ],
                dtype=float,
            )
            mean_train = np.array(
                [
                    d["train_num"] / d["train_den"] if d["train_den"] > 0 else np.nan
                    for d in diag
                ],
                dtype=float,
            )
            acquisition_progress = np.array(
                [
                    (
                        d["progress_num"] / d["progress_den"]
                        if d["progress_den"] > 0
                        else np.nan
                    )
                    for d in diag
                ],
                dtype=float,
            )

        # Broad spectral regions.
        C_low = np.nansum(counts_z_tof * w_low[None, :], axis=1)
        C_high = np.nansum(counts_z_tof * w_high[None, :], axis=1)
        C_diag = np.nansum(counts_z_tof * w_diag[None, :], axis=1)
        C_outside = total_electrons_z - C_diag

        with np.errstate(invalid="ignore", divide="ignore"):
            Y_low = C_low / total_electrons_z
            Y_high = C_high / total_electrons_z
            Y_outside = C_outside / total_electrons_z

        outside_rel = relative_change(Y_outside)

        scan_results.append(
            {
                "scan": iscan,
                "delay_fs": delay_fs,
                "Y_low": Y_low,
                "Y_high": Y_high,
                "Y_outside": Y_outside,
                "outside_rel": outside_rel,
                "electrons_per_shot": electrons_per_shot,
                "mean_gmd": mean_gmd,
                "hor_pos": mean_hor,
                "ver_pos": mean_ver,
                "mpe": mean_mpe,
                "z_std": mean_zstd,
                "tID": mean_tid,
                "train_index": mean_train,
                "acquisition_progress": acquisition_progress,
            }
        )

if len(scan_results) == 0:
    raise ValueError("No complete scans could be reconstructed.")

# Figure 1: Overlay all repeated scans as a function of PHYSICAL DELAY.
fig, axes = plt.subplots(4, 2, figsize=(13, 13), sharex=True, constrained_layout=True)
axes = axes.ravel()

for result in scan_results:
    t = result["delay_fs"]
    label = f"scan {result['scan']}"

    axes[0].plot(t, result["outside_rel"], "o-", ms=2.5, lw=0.9, label=label)
    axes[1].plot(t, result["electrons_per_shot"], "o-", ms=2.5, lw=0.9, label=label)
    axes[2].plot(t, result["mean_gmd"], "o-", ms=2.5, lw=0.9, label=label)
    axes[3].plot(t, result["hor_pos"], "o-", ms=2.5, lw=0.9, label=label)
    axes[4].plot(t, result["ver_pos"], "o-", ms=2.5, lw=0.9, label=label)
    axes[5].plot(t, result["mpe"], "o-", ms=2.5, lw=0.9, label=label)
    axes[6].plot(t, result["z_std"], "o-", ms=2.5, lw=0.9, label=label)
    axes[7].plot(t, result["acquisition_progress"], "o-", ms=2.5, lw=0.9, label=label)

axes[0].axhline(0.0, color="k", lw=0.7, alpha=0.4)
axes[0].set_title("Spectral redistribution")
axes[0].set_ylabel("outside 220-270 eV\nrelative yield (%)")
axes[1].set_title("Electron count")
axes[1].set_ylabel("electrons / shot")
axes[2].set_title("GMD")
axes[2].set_ylabel("mean GMD / shot")
axes[3].set_title("hor_pos")
axes[3].set_ylabel("hor_pos")
axes[4].set_title("ver_pos")
axes[4].set_ylabel("ver_pos")
axes[5].set_title("mpe")
axes[5].set_ylabel("mpe")
axes[6].set_title("z_std")
axes[6].set_ylabel("mean z_std")
axes[7].set_title("Acquisition position within each scan")
axes[7].set_ylabel("scan progress")

for ax in axes:
    ax.set_xlabel("delay (fs)")
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
fig.suptitle(f"{SCAN_NAME}\nDiagnostics versus pump-probe delay", fontsize=13)
plt.show()

# Figure 2: Same quantities against ACTUAL GLOBAL ACQUISITION ORDER.
fig, axes = plt.subplots(4, 2, figsize=(13, 13), sharex=True, constrained_layout=True)
axes = axes.ravel()

for result in scan_results:
    x = result["train_index"]
    label = f"scan {result['scan']}"

    axes[0].plot(x, 100.0 * result["Y_outside"], "o-", ms=2.5, lw=0.9, label=label)
    axes[1].plot(x, result["electrons_per_shot"], "o-", ms=2.5, lw=0.9, label=label)
    axes[2].plot(x, result["mean_gmd"], "o-", ms=2.5, lw=0.9, label=label)
    axes[3].plot(x, result["hor_pos"], "o-", ms=2.5, lw=0.9, label=label)
    axes[4].plot(x, result["ver_pos"], "o-", ms=2.5, lw=0.9, label=label)
    axes[5].plot(x, result["mpe"], "o-", ms=2.5, lw=0.9, label=label)
    axes[6].plot(x, result["z_std"], "o-", ms=2.5, lw=0.9, label=label)
    axes[7].plot(x, result["tID"] - result["tID"][0], "o-", ms=2.5, lw=0.9, label=label)

axes[0].set_title("Spectral redistribution")
axes[0].set_ylabel("outside 220-270 eV\nfraction of electrons (%)")
axes[1].set_title("Electron count")
axes[1].set_ylabel("electrons / shot")
axes[2].set_title("GMD")
axes[2].set_ylabel("mean GMD / shot")
axes[3].set_title("hor_pos")
axes[3].set_ylabel("hor_pos")
axes[4].set_title("ver_pos")
axes[4].set_ylabel("ver_pos")
axes[5].set_title("mpe")
axes[5].set_ylabel("mpe")
axes[6].set_title("z_std")
axes[6].set_ylabel("mean z_std")
axes[7].set_title("Recorded tID within each scan")
axes[7].set_ylabel("tID - first tID")

for ax in axes:
    ax.set_xlabel("global train index / acquisition order")
    ax.grid(alpha=0.2)

axes[0].legend(fontsize=8)
fig.suptitle(f"{SCAN_NAME}\nDiagnostics versus actual acquisition order", fontsize=13)
plt.show()

# Numerical correlations.
diagnostic_names = {
    "electrons_per_shot": "electrons / shot",
    "mean_gmd": "mean GMD / shot",
    "hor_pos": "hor_pos",
    "ver_pos": "ver_pos",
    "mpe": "mpe",
    "z_std": "z_std",
}

print()
print("Correlation diagnostics")
print(
    "(r values are diagnostic only; shared delay dependence can produce correlations without direct causation)"
)
print()

for result in scan_results:
    print(f"scan {result['scan']}:")
    delay = result["delay_fs"]
    outside = result["outside_rel"]

    for key, label in diagnostic_names.items():
        y = result[key]
        good_delay = np.isfinite(delay) & np.isfinite(y)
        good_outside = np.isfinite(outside) & np.isfinite(y)

        r_delay = (
            linregress(delay[good_delay], y[good_delay]).rvalue
            if np.sum(good_delay) >= 3
            else np.nan
        )
        r_outside = (
            linregress(outside[good_outside], y[good_outside]).rvalue
            if np.sum(good_outside) >= 3
            else np.nan
        )

        print(
            f"  {label:18s} r(delay)={r_delay:+.3f}   r(outside yield)={r_outside:+.3f}"
        )
    print()


### Kinetic-energy location of the redistribution

In [ ]:
# For one selected run:
#   1. reconstruct the individual repeated delay scans from the raw H5;
#   2. calculate the electron-yield-normalised spectrum at every delay;
#   3. fit a linear delay slope independently at every KE bin;
#   4. compare the slope from each repeated scan;
#   5. calculate a more robust version in 5 eV-wide KE windows;
#   6. print which coarse KE regions show the strongest positive/negative slope.
# The fitted quantity is: d/dt [ relative electron yield at energy E ]   [% / fs]
# Positive slope = that part of the spectrum gains fractional yield with delay.
# Negative slope = it loses fractional yield with delay.
# No residual-gas subtraction. No smoothing. No interpolation. No detrending. No Jacobian.

# User choice.
SCAN_NAME = "glycine_delay_scan_150C_274.5eV"
COMBINED_DIR = Path(config.COMBINED_DIR)
RAW_H5 = COMBINED_DIR / f"{SCAN_NAME}.h5"
AGG_KEY = f"{SCAN_NAME}_aggregates_tr_etof1.h5"

PLOT_KE_RANGE_EV = (
    0.0,
    280.0,
)  # Plot the physically relevant calibrated part of the spectrum.
COARSE_KE_WIDTH_EV = 5.0  # Stable coarse-window diagnostic.

# Suppress individual KE bins with essentially no statistics.
# This affects only the fine-bin slope plot, not the coarse-window analysis.
MIN_MEAN_COUNTS_PER_DELAY = 5.0
MIN_UNIQUE_DELAY_BINS = 10
RESET_MIN_STEPS = 10.0

# Check inputs and obtain aggregate axes.
if not RAW_H5.exists():
    raise FileNotFoundError(f"Combined H5 not found:\n{RAW_H5}")

if AGG_KEY not in loaded_aggs:
    raise KeyError(f"{AGG_KEY!r} is not present in loaded_aggs.")

agg_scan = loaded_aggs[AGG_KEY]["agg"]
z_edges = np.asarray(agg_scan.z_edges, dtype=float)
tof_edges = np.asarray(agg_scan.tof_edges, dtype=float)
z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
delay_cent_fs_all = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

# KE calibration.
ke_edges = tof_to_ke(tof_edges)
ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])
valid_ke = np.isfinite(ke_edges[:-1]) & np.isfinite(ke_edges[1:]) & np.isfinite(ke_cent)


def ke_window_weights(ke_min, ke_max):
    """Fractional overlap of each calibrated TOF bin with a KE interval."""
    e0 = ke_edges[:-1]
    e1 = ke_edges[1:]
    bin_lo = np.minimum(e0, e1)
    bin_hi = np.maximum(e0, e1)
    width = bin_hi - bin_lo
    overlap = np.maximum(0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min))
    weights = np.zeros_like(width, dtype=float)
    good = valid_ke & (width > 0)
    weights[good] = overlap[good] / width[good]
    return weights


def relative_change(y):
    """Relative change about the delay-averaged value."""
    y = np.asarray(y, dtype=float)
    mean = np.nanmean(y)
    with np.errstate(invalid="ignore", divide="ignore"):
        return 100.0 * (y - mean) / mean


# Read raw delay coordinate.
with h5py.File(RAW_H5, "r") as f:
    z = np.asarray(f["z"][:], dtype=float)
    etof_shape = f["tofs_e"].shape

print("z shape:     ", z.shape)
print("tofs_e shape:", etof_shape)

if z.shape != etof_shape[:2]:
    raise ValueError("z dimensions do not match the first two tofs_e dimensions.")

# Assign shots to the aggregate delay bins.
delay_bin_pulse = np.searchsorted(z_edges, z, side="right") - 1
valid_delay_pulse = (
    np.isfinite(z) & (delay_bin_pulse >= 0) & (delay_bin_pulse < len(z_cent))
)

# Representative delay bin for every train.
n_trains = z.shape[0]
train_delay_bin = np.full(n_trains, -1, dtype=int)
train_delay_purity = np.full(n_trains, np.nan, dtype=float)

for itr in range(n_trains):
    bins = delay_bin_pulse[itr, valid_delay_pulse[itr]]
    if bins.size == 0:
        continue
    values, counts = np.unique(bins, return_counts=True)
    imax = np.argmax(counts)
    train_delay_bin[itr] = values[imax]
    train_delay_purity[itr] = counts[imax] / bins.size

valid_train = train_delay_bin >= 0
print()
print("Valid trains:", np.sum(valid_train), "/", n_trains)
print(
    "Median train-delay agreement:",
    f"{100*np.nanmedian(train_delay_purity[valid_train]):.2f}%",
)

# Build contiguous delay-point visits.
valid_train_indices = np.where(valid_train)[0]
blocks = []
start_train = valid_train_indices[0]
previous_train = valid_train_indices[0]
current_bin = train_delay_bin[start_train]

for itr in valid_train_indices[1:]:
    same_delay = train_delay_bin[itr] == current_bin
    contiguous = itr == previous_train + 1
    if not same_delay or not contiguous:
        blocks.append(
            {
                "start": start_train,
                "stop": previous_train + 1,
                "delay_bin": int(current_bin),
            }
        )
        start_train = itr
        current_bin = train_delay_bin[itr]
    previous_train = itr

blocks.append(
    {
        "start": start_train,
        "stop": previous_train + 1,
        "delay_bin": int(current_bin),
    }
)

block_bins = np.array([block["delay_bin"] for block in blocks], dtype=int)
block_delay_fs = delay_cent_fs_all[block_bins]

# Detect repeated increasing-delay scans.
ddelay = np.diff(block_delay_fs)
nonzero_steps = np.abs(ddelay[np.isfinite(ddelay) & (np.abs(ddelay) > 0)])

if nonzero_steps.size == 0:
    raise ValueError("Could not determine the delay step.")

typical_step = np.nanmedian(nonzero_steps)
ordinary_steps = ddelay[
    np.isfinite(ddelay) & (np.abs(ddelay) > 0) & (np.abs(ddelay) <= 2.0 * typical_step)
]
n_positive = np.sum(ordinary_steps > 0)
n_negative = np.sum(ordinary_steps < 0)

if n_positive > n_negative:
    dominant_direction = 1.0
elif n_negative > n_positive:
    dominant_direction = -1.0
else:
    raise ValueError("Delay scan direction is ambiguous.")

scan_starts = [0]
for i, step in enumerate(ddelay):
    if not np.isfinite(step):
        continue
    is_large_jump = abs(step) >= RESET_MIN_STEPS * typical_step
    is_reset = np.sign(step) == -dominant_direction
    if is_large_jump and is_reset:
        scan_starts.append(i + 1)

scan_starts = sorted(set(scan_starts))
scan_stops = scan_starts[1:] + [len(blocks)]
scan_block_ranges_all = list(zip(scan_starts, scan_stops))

scan_block_ranges = []
for ib0, ib1 in scan_block_ranges_all:
    n_unique = len(np.unique(block_bins[ib0:ib1]))
    if n_unique >= MIN_UNIQUE_DELAY_BINS:
        scan_block_ranges.append((ib0, ib1))

print("Delay-point visits:", len(blocks))
print("Typical delay step:", f"{typical_step:.4f} fs")
print("Complete scans:", len(scan_block_ranges))
print()

for iscan, (ib0, ib1) in enumerate(scan_block_ranges, start=1):
    delay_bins_scan = block_bins[ib0:ib1]
    delays = block_delay_fs[ib0:ib1]
    print(
        f"scan {iscan}: {len(delays)} visits, {len(np.unique(delay_bins_scan))} unique bins, {delays[0]:.3f} -> {delays[-1]:.3f} fs"
    )

# Reconstruct spectra independently for each repeated scan.
scan_results = []

with h5py.File(RAW_H5, "r") as f:
    etof_ds = f["tofs_e"]

    for iscan, (ib0, ib1) in enumerate(scan_block_ranges, start=1):
        spectra_by_delay = {}
        shots_by_delay = {}

        for block in blocks[ib0:ib1]:
            delay_bin = block["delay_bin"]
            start = block["start"]
            stop = block["stop"]

            etof_chunk = np.asarray(etof_ds[start:stop])
            pulse_delay_bins = delay_bin_pulse[start:stop]
            valid_shots = valid_delay_pulse[start:stop] & (
                pulse_delay_bins == delay_bin
            )
            n_valid_shots = int(np.sum(valid_shots))

            if n_valid_shots == 0:
                continue

            events = etof_chunk[valid_shots].ravel()
            events = events[events > 0]  # Remove zero padding.
            events = events[
                (events >= tof_edges[0]) & (events < tof_edges[-1])
            ]  # Keep only configured range.

            hist, _ = np.histogram(events, bins=tof_edges)

            if delay_bin not in spectra_by_delay:
                spectra_by_delay[delay_bin] = np.zeros(len(tof_edges) - 1, dtype=float)
                shots_by_delay[delay_bin] = 0

            spectra_by_delay[delay_bin] += hist
            shots_by_delay[delay_bin] += n_valid_shots

        delay_bins_scan = np.array(sorted(spectra_by_delay), dtype=int)
        counts_z_tof = np.stack([spectra_by_delay[i] for i in delay_bins_scan], axis=0)
        delay_fs = delay_cent_fs_all[delay_bins_scan]
        total_electrons_z = np.nansum(counts_z_tof, axis=1)

        # Normalised spectrum.
        with np.errstate(invalid="ignore", divide="ignore"):
            spectrum_fraction = counts_z_tof / total_electrons_z[:, None]

        mean_fraction = np.nanmean(spectrum_fraction, axis=0)
        mean_counts = np.nanmean(counts_z_tof, axis=0)

        with np.errstate(invalid="ignore", divide="ignore"):
            rel_map = (
                100.0
                * (spectrum_fraction - mean_fraction[None, :])
                / mean_fraction[None, :]
            )

        # Linear slope at EVERY native TOF/KE bin.
        slope_native = np.full(len(tof_edges) - 1, np.nan, dtype=float)
        valid_slope_bins = (
            valid_ke
            & np.isfinite(mean_fraction)
            & (mean_fraction > 0)
            & (mean_counts >= MIN_MEAN_COUNTS_PER_DELAY)
        )

        for ibin in np.where(valid_slope_bins)[0]:
            y = rel_map[:, ibin]
            good = np.isfinite(delay_fs) & np.isfinite(y)
            if np.sum(good) < 3:
                continue
            fit = linregress(delay_fs[good], y[good])
            slope_native[ibin] = fit.slope

        scan_results.append(
            {
                "scan": iscan,
                "delay_fs": delay_fs,
                "counts_z_tof": counts_z_tof,
                "total_electrons": total_electrons_z,
                "mean_fraction": mean_fraction,
                "mean_counts": mean_counts,
                "slope_native": slope_native,
            }
        )

# Reorder native spectrum in increasing KE.
plot_native_mask = (
    valid_ke & (ke_cent >= PLOT_KE_RANGE_EV[0]) & (ke_cent <= PLOT_KE_RANGE_EV[1])
)
native_idx = np.where(plot_native_mask)[0]

if native_idx.size == 0:
    raise ValueError("No calibrated bins in the selected KE plotting range.")

ke_native_plot = ke_cent[native_idx]
sort_idx = np.argsort(ke_native_plot)
ke_native_plot = ke_native_plot[sort_idx]

native_slopes = np.stack(
    [r["slope_native"][native_idx][sort_idx] for r in scan_results], axis=0
)
native_mean_slope = np.nanmean(native_slopes, axis=0)
native_std_slope = np.nanstd(native_slopes, axis=0, ddof=1)

# Mean spectrum across scans, for orientation.
mean_spectra = np.stack(
    [r["mean_fraction"][native_idx][sort_idx] for r in scan_results], axis=0
)
mean_spectrum_plot = np.nanmean(mean_spectra, axis=0)

if np.nanmax(mean_spectrum_plot) > 0:
    mean_spectrum_plot = mean_spectrum_plot / np.nanmax(mean_spectrum_plot)

# Coarse 5-eV-window slopes.
coarse_edges = np.arange(
    PLOT_KE_RANGE_EV[0], PLOT_KE_RANGE_EV[1] + COARSE_KE_WIDTH_EV, COARSE_KE_WIDTH_EV
)
coarse_cent = 0.5 * (coarse_edges[:-1] + coarse_edges[1:])
n_coarse = len(coarse_cent)
coarse_slopes = np.full((len(scan_results), n_coarse), np.nan, dtype=float)

for iscan, result in enumerate(scan_results):
    delay_fs = result["delay_fs"]
    counts_z_tof = result["counts_z_tof"]
    total_electrons_z = result["total_electrons"]

    for iw in range(n_coarse):
        e_min = coarse_edges[iw]
        e_max = coarse_edges[iw + 1]
        weights = ke_window_weights(e_min, e_max)
        C_window = np.nansum(counts_z_tof * weights[None, :], axis=1)

        with np.errstate(invalid="ignore", divide="ignore"):
            Y_window = C_window / total_electrons_z

        if np.nanmean(C_window) < MIN_MEAN_COUNTS_PER_DELAY:
            continue

        Y_rel = relative_change(Y_window)
        good = np.isfinite(delay_fs) & np.isfinite(Y_rel)
        if np.sum(good) < 3:
            continue

        fit = linregress(delay_fs[good], Y_rel[good])
        coarse_slopes[iscan, iw] = fit.slope

coarse_mean = np.nanmean(coarse_slopes, axis=0)
coarse_std = np.nanstd(coarse_slopes, axis=0, ddof=1)

# Plot.
fig, axes = plt.subplots(3, 1, figsize=(12, 11), sharex=True, constrained_layout=True)

# 1. Mean spectrum.
axes[0].plot(ke_native_plot, mean_spectrum_plot, lw=1.0)
axes[0].set_ylabel("mean spectrum\n(normalised)")
axes[0].set_title("Delay-averaged electron spectrum")

# 2. Native-bin spectral slope.
for iscan, slopes in enumerate(native_slopes, start=1):
    axes[1].plot(ke_native_plot, slopes, lw=0.8, alpha=0.45, label=f"scan {iscan}")

axes[1].plot(ke_native_plot, native_mean_slope, lw=1.8, label="mean")
axes[1].fill_between(
    ke_native_plot,
    native_mean_slope - native_std_slope,
    native_mean_slope + native_std_slope,
    alpha=0.2,
    label="+/-1 scan SD",
)
axes[1].axhline(0.0, color="k", lw=0.8)
axes[1].set_ylabel("spectral slope\n(% / fs)")
axes[1].set_title("Linear delay slope at native calibrated eTOF bins")
axes[1].legend(fontsize=8, ncol=4)

# 3. Coarse 5-eV slope.
for iscan in range(len(scan_results)):
    axes[2].plot(coarse_cent, coarse_slopes[iscan], "o-", ms=3, lw=0.8, alpha=0.45)

axes[2].errorbar(
    coarse_cent,
    coarse_mean,
    yerr=coarse_std,
    fmt="o-",
    ms=4,
    lw=1.5,
    capsize=2,
    label="mean +/- scan SD",
)
axes[2].axhline(0.0, color="k", lw=0.8)
axes[2].set_ylabel(f"{COARSE_KE_WIDTH_EV:.0f} eV-window slope\n(% / fs)")
axes[2].set_xlabel("electron kinetic energy (eV)")
axes[2].set_title(f"Integrated {COARSE_KE_WIDTH_EV:.0f} eV windows")
axes[2].legend(fontsize=8)

# Mark the diagnostic region and the broad thesis windows.
for ax in axes:
    ax.axvline(220.0, color="k", ls=":", lw=0.8, alpha=0.5)
    ax.axvline(270.0, color="k", ls=":", lw=0.8, alpha=0.5)
    ax.axvline(240.0, color="k", ls="--", lw=0.6, alpha=0.35)
    ax.axvline(245.0, color="k", ls="--", lw=0.6, alpha=0.35)
    ax.set_xlim(PLOT_KE_RANGE_EV)
    ax.grid(alpha=0.2)

fig.suptitle(
    f"{SCAN_NAME}\nWhere does the delay-dependent spectral redistribution occur?",
    fontsize=13,
)
plt.show()

# Numerical summary of coarse windows.
print("\nCoarse-window spectral slopes")
print(f"({COARSE_KE_WIDTH_EV:.0f} eV windows; mean +/- SD across repeated scans)\n")

for iw in range(n_coarse):
    if not np.isfinite(coarse_mean[iw]):
        continue
    print(
        f"{coarse_edges[iw]:6.1f}-{coarse_edges[iw + 1]:6.1f} eV: {coarse_mean[iw]:+.4f} +/- {coarse_std[iw]:.4f} %/fs"
    )

# Identify strongest reproducible changes outside 220-270 eV.
outside_mask = (coarse_cent < 220.0) | (coarse_cent > 270.0)
valid_outside = outside_mask & np.isfinite(coarse_mean)

if np.any(valid_outside):
    outside_indices = np.where(valid_outside)[0]
    i_positive = outside_indices[np.nanargmax(coarse_mean[outside_indices])]
    i_negative = outside_indices[np.nanargmin(coarse_mean[outside_indices])]

    print("\nStrongest positive slope outside 220-270 eV:")
    print(
        f"  {coarse_edges[i_positive]:.1f}-{coarse_edges[i_positive + 1]:.1f} eV: {coarse_mean[i_positive]:+.4f} +/- {coarse_std[i_positive]:.4f} %/fs"
    )
    print("\nStrongest negative slope outside 220-270 eV:")
    print(
        f"  {coarse_edges[i_negative]:.1f}-{coarse_edges[i_negative + 1]:.1f} eV: {coarse_mean[i_negative]:+.4f} +/- {coarse_std[i_negative]:.4f} %/fs"
    )


### Residual-gas and low-kinetic-energy diagnostic

In [ ]:
# INVESTIGATE WHETHER RESIDUAL GAS PRODUCES THE DELAY-DEPENDENT GRADIENT
# This script:
#   1. Loads the selected glycine delay-scan aggregate;
#   2. Loads the measured 274.0 eV residual-gas aggregate directly from HDF5;
#   3. Compares their low-KE spectra shapes;
#   4. Tracks raw electrons/shot in key target windows (0-15, 15-30, 30-50, 220-270 eV);
#   5. Tracks the ABSOLUTE electron fraction in those same windows;
#   6. Calculates raw-count and absolute-fraction slopes via linear regression;
#   7. Compares the early-to-late raw spectral difference with the background shape;
#   8. Computes 5 eV fine-window slopes in absolute yield and percentage points.
# STRICT CONSTRAINTS: Zero background subtraction, smoothing, interpolation,
# detrending, or Jacobian application. The background KE axis is shifted purely
# on the plotting x-axis by the photon energy difference (+0.5 eV).

# Configuration parameters
SCAN_NAME = "glycine_delay_scan_150C_274.5eV"
AGG_KEY = f"{SCAN_NAME}_aggregates_tr_etof1.h5"
BACKGROUND_FILENAME = "background_274.5eV_run58890_aggregates_etof1.h5"
BACKGROUND_PHOTON_ENERGY_EV = 274.0
PROCESSED_DIR = Path(config.COMBINED_DIR).parent

# Window definitions
AR_BAND_KE_EV = (15.0, 30.0)
LOW_KE_RANGE_EV = (0.0, 50.0)
GLYCINE_RANGE_EV = (220.0, 270.0)

N_EDGE_DELAY_BINS = 10
COARSE_KE_RANGE_EV = (0.0, 270.0)
COARSE_KE_WIDTH_EV = 5.0

# 1. Load Glycine Target Aggregate
if AGG_KEY not in loaded_aggs:
    raise KeyError(f"{AGG_KEY!r} is missing from loaded_aggs.")

signal_item = loaded_aggs[AGG_KEY]
signal_energy = float(signal_item["energy"])
agg = signal_item["agg"]

print(f"Signal run: {AGG_KEY}")
print(f"Signal photon energy: {signal_energy:.1f} eV")

# 2. Locate Residual-Gas Aggregate
background_matches = list(PROCESSED_DIR.rglob(BACKGROUND_FILENAME))
if not background_matches:
    raise FileNotFoundError(
        f"Could not find {BACKGROUND_FILENAME!r} below:\n{PROCESSED_DIR}"
    )
if len(background_matches) > 1:
    raise RuntimeError(
        "Multiple background matching files detected:\n"
        + "\n".join(str(p) for p in background_matches)
    )

BACKGROUND_H5 = background_matches[0]
print("Background file found:", BACKGROUND_H5)


# Helper Functions
def make_ke_weights(tof_edges_local, ke_min, ke_max):
    """Computes exact fractional energy bin overlap coverage matrices."""
    ke_edges_local = tof_to_ke(np.asarray(tof_edges_local, dtype=float))
    e0, e1 = ke_edges_local[:-1], ke_edges_local[1:]

    valid = np.isfinite(e0) & np.isfinite(e1)
    bin_lo = np.minimum(e0, e1)
    bin_hi = np.maximum(e0, e1)
    width = bin_hi - bin_lo

    overlap = np.maximum(0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min))
    weights = np.zeros_like(width, dtype=float)
    good = valid & (width > 0)
    weights[good] = overlap[good] / width[good]
    return weights


def fit_slope(x, y):
    """Applies clean linear standard regression over valid numbers."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    if np.sum(good) < 3:
        return None
    return linregress(x[good], y[good])


# Process Raw Signal Arrays
D = np.asarray(agg.D, dtype=float)
n = np.asarray(agg.n_per_bin, dtype=float)
tof_edges = np.asarray(agg.tof_edges, dtype=float)
z_edges = np.asarray(agg.z_edges, dtype=float)

if D.ndim != 3 or D.shape[:2] != n.shape or D.shape[2] != len(tof_edges) - 1:
    raise ValueError("Signal data multi-axis shape mismatch detected.")

# Collapse dimensions to get absolute metrics
counts_z_tof = np.nansum(D * n[..., None], axis=0)
shots_z = np.nansum(n, axis=0)
total_electrons_z = np.nansum(counts_z_tof, axis=1)

with np.errstate(invalid="ignore", divide="ignore"):
    counts_per_shot_z_tof = counts_z_tof / shots_z[:, None]
    electrons_per_shot_z = total_electrons_z / shots_z
    spectrum_fraction_z_tof = counts_z_tof / total_electrons_z[:, None]

z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
delay_fs = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

ke_edges = tof_to_ke(tof_edges)
ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])
valid_ke = np.isfinite(ke_edges[:-1]) & np.isfinite(ke_edges[1:]) & np.isfinite(ke_cent)

# Load and Process Residual-Gas Arrays
with h5py.File(BACKGROUND_H5, "r") as f:
    bg_D = np.asarray(f["D"][:], dtype=float)
    bg_n = np.asarray(f["n_per_bin"][:], dtype=float)
    bg_tof_edges = np.asarray(f["tof_edges"][:], dtype=float)

print(f"\nBackground D shape: {bg_D.shape} | Background n shape: {bg_n.shape}")

if bg_D.ndim == 2 and bg_n.ndim == 1:
    bg_counts_tof = np.nansum(bg_D * bg_n[:, None], axis=0)
    bg_shots = np.nansum(bg_n)
elif bg_D.ndim == 3 and bg_n.ndim == 2:
    bg_counts_tof = np.nansum(bg_D * bg_n[..., None], axis=(0, 1))
    bg_shots = np.nansum(bg_n)
else:
    raise ValueError(f"Unexpected background shapes: D={bg_D.shape}, n={bg_n.shape}")

with np.errstate(invalid="ignore", divide="ignore"):
    bg_counts_per_shot = bg_counts_tof / bg_shots

bg_total_electrons = np.nansum(bg_counts_tof)
with np.errstate(invalid="ignore", divide="ignore"):
    bg_fraction_tof = bg_counts_tof / bg_total_electrons

bg_ke_edges = tof_to_ke(bg_tof_edges)
bg_ke_cent = 0.5 * (bg_ke_edges[:-1] + bg_ke_edges[1:])
bg_valid_ke = (
    np.isfinite(bg_ke_edges[:-1])
    & np.isfinite(bg_ke_edges[1:])
    & np.isfinite(bg_ke_cent)
)

# Apply rigid plotting-axis shift to simulate shift under energy differences
BACKGROUND_KE_PLOT_SHIFT_EV = signal_energy - BACKGROUND_PHOTON_ENERGY_EV
bg_ke_cent_shifted = bg_ke_cent + BACKGROUND_KE_PLOT_SHIFT_EV

print(f"Background KE plotting shift: {BACKGROUND_KE_PLOT_SHIFT_EV:+.3f} eV")
print("(x-axis shift only; background counts remain unchanged)")

# Evaluate Diagnostic Windows
windows = {
    "0-15 eV": (0.0, 15.0),
    "15-30 eV": (15.0, 30.0),
    "30-50 eV": (30.0, 50.0),
    "0-50 eV": (0.0, 50.0),
    "220-270 eV": (220.0, 270.0),
}

window_results = {}
for name, (e_min, e_max) in windows.items():
    weights = make_ke_weights(tof_edges, e_min, e_max)
    counts_window = np.nansum(counts_z_tof * weights[None, :], axis=1)

    with np.errstate(invalid="ignore", divide="ignore"):
        counts_per_shot = counts_window / shots_z
        fraction = counts_window / total_electrons_z

    raw_fit = fit_slope(delay_fs, counts_per_shot)
    fraction_fit = fit_slope(delay_fs, 100.0 * fraction)

    window_results[name] = {
        "counts": counts_window,
        "counts_per_shot": counts_per_shot,
        "fraction": fraction,
        "raw_fit": raw_fit,
        "fraction_fit": fraction_fit,
    }

bg_window_results = {}
for name, (e_min, e_max) in windows.items():
    bg_e_min = e_min - BACKGROUND_KE_PLOT_SHIFT_EV
    bg_e_max = e_max - BACKGROUND_KE_PLOT_SHIFT_EV

    weights = make_ke_weights(bg_tof_edges, bg_e_min, bg_e_max)
    bg_counts_window = np.nansum(bg_counts_tof * weights)

    bg_window_results[name] = {
        "counts": bg_counts_window,
        "fraction": bg_counts_window / bg_total_electrons,
        "counts_per_shot": bg_counts_window / bg_shots,
    }

# Calculate Early vs Late Raw Signal Changes
order = np.argsort(delay_fs)
n_edge = min(N_EDGE_DELAY_BINS, len(order) // 3)
if n_edge < 2:
    raise ValueError("Insufficient delay coordinates for edge boundary comparisons.")

early_idx = order[:n_edge]
late_idx = order[-n_edge:]

early_counts_per_shot = np.nanmean(counts_per_shot_z_tof[early_idx], axis=0)
late_counts_per_shot = np.nanmean(counts_per_shot_z_tof[late_idx], axis=0)
delta_counts_per_shot = late_counts_per_shot - early_counts_per_shot

print(
    f"\nEarly delay range: {np.nanmin(delay_fs[early_idx]):.3f} to {np.nanmax(delay_fs[early_idx]):.3f} fs"
)
print(
    f"Late delay range:  {np.nanmin(delay_fs[late_idx]):.3f} to {np.nanmax(delay_fs[late_idx]):.3f} fs"
)

# FIGURE 1: Analytical Window Profiles & Gas Shape Comparison
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)

# A. Normalised Low-KE Spectral Contrast
signal_mean_counts_per_shot = np.nanmean(counts_per_shot_z_tof, axis=0)
signal_low_mask = (
    valid_ke & (ke_cent >= LOW_KE_RANGE_EV[0]) & (ke_cent <= LOW_KE_RANGE_EV[1])
)
bg_low_mask = (
    bg_valid_ke
    & (bg_ke_cent_shifted >= LOW_KE_RANGE_EV[0])
    & (bg_ke_cent_shifted <= LOW_KE_RANGE_EV[1])
)

signal_shape = signal_mean_counts_per_shot[signal_low_mask]
bg_shape = bg_counts_per_shot[bg_low_mask]

if np.nanmax(signal_shape) > 0:
    signal_shape /= np.nanmax(signal_shape)
if np.nanmax(bg_shape) > 0:
    bg_shape /= np.nanmax(bg_shape)

axes[0, 0].plot(ke_cent[signal_low_mask], signal_shape, lw=1.2, label="glycine run")
axes[0, 0].plot(
    bg_ke_cent_shifted[bg_low_mask],
    bg_shape,
    lw=1.2,
    label=f"measured background (+{BACKGROUND_KE_PLOT_SHIFT_EV:.1f} eV shift)",
)
axes[0, 0].axvspan(*AR_BAND_KE_EV, alpha=0.12, label="15-30 eV test region")
axes[0, 0].set_xlim(LOW_KE_RANGE_EV)
axes[0, 0].set_xlabel("electron kinetic energy (eV)")
axes[0, 0].set_ylabel("spectrum / maximum")
axes[0, 0].set_title("Low-KE Spectral Shape Match")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.2)

# B. Raw Yield Tracks
for name in ["15-30 eV", "30-50 eV", "220-270 eV"]:
    axes[0, 1].plot(
        delay_fs,
        window_results[name]["counts_per_shot"],
        "o-",
        ms=3,
        lw=1.0,
        label=name,
    )
axes[0, 1].set_xlabel("delay (fs)")
axes[0, 1].set_ylabel("electrons / shot")
axes[0, 1].set_title("Raw Yield Tracks (Before Normalisation)")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.2)

# C. Absolute Yield Fractions
for name in ["15-30 eV", "30-50 eV", "220-270 eV"]:
    axes[1, 0].plot(
        delay_fs,
        100.0 * window_results[name]["fraction"],
        "o-",
        ms=3,
        lw=1.0,
        label=name,
    )
axes[1, 0].set_xlabel("delay (fs)")
axes[1, 0].set_ylabel("fraction of all detected electrons (%)")
axes[1, 0].set_title("Absolute Total Yield Contribution")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.2)

# D. Dynamic Component Evolution Shape Analysis
delta_low = delta_counts_per_shot[signal_low_mask]
delta_scale = np.nanmax(np.abs(delta_low))
delta_low_norm = (
    delta_low / delta_scale
    if (np.isfinite(delta_scale) and delta_scale > 0)
    else delta_low
)

axes[1, 1].plot(
    ke_cent[signal_low_mask],
    delta_low_norm,
    lw=1.2,
    label="late - early raw electrons/shot",
)
axes[1, 1].plot(
    bg_ke_cent_shifted[bg_low_mask],
    bg_shape,
    lw=1.2,
    label="measured residual-gas shape",
)
axes[1, 1].axhline(0.0, color="k", lw=0.8)
axes[1, 1].axvspan(*AR_BAND_KE_EV, alpha=0.12)
axes[1, 1].set_xlim(LOW_KE_RANGE_EV)
axes[1, 1].set_xlabel("electron kinetic energy (eV)")
axes[1, 1].set_ylabel("normalised shape")
axes[1, 1].set_title("Does the growing component resemble residual gas?")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.2)

fig.suptitle(
    f"{SCAN_NAME}\nResidual-gas investigation - no background subtraction", fontsize=13
)
plt.show()

# 8. Fine Coarse 5 eV Window Slopes Analysis
coarse_edges = np.arange(
    COARSE_KE_RANGE_EV[0],
    COARSE_KE_RANGE_EV[1] + COARSE_KE_WIDTH_EV,
    COARSE_KE_WIDTH_EV,
)
coarse_cent = 0.5 * (coarse_edges[:-1] + coarse_edges[1:])

raw_slopes = np.full(len(coarse_cent), np.nan, dtype=float)
fraction_slopes = np.full(len(coarse_cent), np.nan, dtype=float)

for iw in range(len(coarse_cent)):
    e_min, e_max = coarse_edges[iw], coarse_edges[iw + 1]
    weights = make_ke_weights(tof_edges, e_min, e_max)
    C = np.nansum(counts_z_tof * weights[None, :], axis=1)

    with np.errstate(invalid="ignore", divide="ignore"):
        C_per_shot = C / shots_z
        fraction = C / total_electrons_z

    raw_fit = fit_slope(delay_fs, C_per_shot)
    fraction_fit = fit_slope(delay_fs, 100.0 * fraction)

    if raw_fit is not None:
        raw_slopes[iw] = raw_fit.slope
    if fraction_fit is not None:
        fraction_slopes[iw] = fraction_fit.slope

# FIGURE 2: Continuous Slope Distributions Across KE
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, constrained_layout=True)

axes[0].plot(coarse_cent, raw_slopes, "o-", ms=4, lw=1.0)
axes[0].axhline(0.0, color="k", lw=0.8)
axes[0].axvspan(*AR_BAND_KE_EV, alpha=0.12)
axes[0].axvspan(*GLYCINE_RANGE_EV, alpha=0.08)
axes[0].set_ylabel("raw yield slope\n(electrons / shot / fs)")
axes[0].set_title("Does the absolute electron count change at low KE?")
axes[0].grid(alpha=0.2)

axes[1].plot(coarse_cent, fraction_slopes, "o-", ms=4, lw=1.0)
axes[1].axhline(0.0, color="k", lw=0.8)
axes[1].axvspan(*AR_BAND_KE_EV, alpha=0.12, label="15-30 eV")
axes[1].axvspan(*GLYCINE_RANGE_EV, alpha=0.08, label="220-270 eV")
axes[1].set_xlabel("electron kinetic energy (eV)")
axes[1].set_ylabel("absolute fraction slope\n(percentage points / fs)")
axes[1].set_title("Where is the total electron fraction being redistributed?")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.2)

fig.suptitle(
    f"{SCAN_NAME}\nRaw versus normalisation-induced spectral changes", fontsize=13
)
plt.show()

# Quantitative Statistical Summary Prints
print("\n" + "=" * 70)
print("KEY REGION DIAGNOSTIC WINDOW SLOPES")
print("=" * 70)

for name in ["0-15 eV", "15-30 eV", "30-50 eV", "0-50 eV", "220-270 eV"]:
    raw_fit = window_results[name]["raw_fit"]
    frac_fit = window_results[name]["fraction_fit"]
    bg_result = bg_window_results[name]

    print(f"\n[{name.upper()}]")
    if raw_fit is not None:
        print(
            f"  Raw yields/shot slope: {raw_fit.slope:+.6e} /fs | r={raw_fit.rvalue:+.3f}, p={raw_fit.pvalue:.3g}"
        )
    if frac_fit is not None:
        print(
            f"  Absolute fraction slope: {frac_fit.slope:+.6f} pp/fs  | r={frac_fit.rvalue:+.3f}, p={frac_fit.pvalue:.3g}"
        )
    print(
        f"  Gas composition fraction: {100*bg_result['fraction']:.2f}% of static residual profile"
    )

# Normalisation Balances Tracking
low_fraction_slope = window_results["15-30 eV"]["fraction_fit"].slope
gly_fraction_slope = window_results["220-270 eV"]["fraction_fit"].slope

print("\n" + "=" * 70)
print("NORMALISATION REDISTRIBUTION ACCOUNTING")
print("=" * 70)
print(
    f"  15-30 eV absolute fraction slope:  {low_fraction_slope:+.6f} percentage points/fs"
)
print(
    f"  220-270 eV absolute fraction slope: {gly_fraction_slope:+.6f} percentage points/fs"
)

if low_fraction_slope > 0 and gly_fraction_slope < 0:
    balance_ratio = low_fraction_slope / abs(gly_fraction_slope)
    print(
        f"  --> Compensation ratio (15-30 eV gain / 220-270 eV loss): {balance_ratio:.3f}"
    )

# Extract highest local trends below 50 eV boundary
valid_low = (coarse_cent < 50.0) & np.isfinite(raw_slopes)
if np.any(valid_low):
    idx_low = np.where(valid_low)[0]
    order_low = idx_low[np.argsort(raw_slopes[idx_low])[::-1]]
    print("\nLargest absolute raw counts/shot growth zones below 50 eV:")
    for iw in order_low[:5]:
        print(
            f"  {coarse_edges[iw]:.1f}-{coarse_edges[iw + 1]:.1f} eV: {raw_slopes[iw]:+.6e} electrons/shot/fs"
        )
print("=" * 70 + "\n")


### Normalisation-denominator check

In [ ]:
# Test whether low-KE electrons create the apparent high-KE gradient

# User choice and environment setup hooks
SCAN_NAME = "glycine_delay_scan_150C_274.5eV"
AGG_KEY = f"{SCAN_NAME}_aggregates_tr_etof1.h5"

LOW_WINDOW_KE_EV = (225.0, 240.0)
HIGH_WINDOW_KE_EV = (245.0, 266.0)
PLOT_KE_RANGE_EV = (220.0, 270.0)

# Exclude the low-energy contribution from the electron-yield denominator.
NORMALISATION_MIN_KE_EV = 50.0

# Extract aggregate properties
if AGG_KEY not in loaded_aggs:
    raise KeyError(f"{AGG_KEY!r} is not present in loaded_aggs.")

agg = loaded_aggs[AGG_KEY]["agg"]

D = np.asarray(agg.D, dtype=float)
n = np.asarray(agg.n_per_bin, dtype=float)
tof_edges = np.asarray(agg.tof_edges, dtype=float)
z_edges = np.asarray(agg.z_edges, dtype=float)

if D.ndim != 3:
    raise ValueError(f"Expected D[gmd, delay, tof], got {D.shape}.")

if D.shape[:2] != n.shape:
    raise ValueError(
        f"D leading shape {D.shape[:2]} does not match n_per_bin {n.shape}."
    )

# Reconstruct raw metrics
counts_z_tof = np.nansum(D * n[..., None], axis=0)
shots_z = np.nansum(n, axis=0)
total_electrons_z = np.nansum(counts_z_tof, axis=1)

# Build coordinate axes
z_cent = 0.5 * (z_edges[:-1] + z_edges[1:])
delay_fs = (z_cent - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT
delay_edges_fs = (z_edges - DELAY_ZERO_STAGE) * FS_PER_STAGE_UNIT

ke_edges = tof_to_ke(tof_edges)
ke_cent = 0.5 * (ke_edges[:-1] + ke_edges[1:])

valid_ke = np.isfinite(ke_edges[:-1]) & np.isfinite(ke_edges[1:]) & np.isfinite(ke_cent)


# Geometry and mathematical helper functions
def ke_window_weights(ke_min, ke_max):
    """Calculates the exact percentage fractional coverage of raw energy bins."""
    e0 = ke_edges[:-1]
    e1 = ke_edges[1:]

    bin_lo = np.minimum(e0, e1)
    bin_hi = np.maximum(e0, e1)
    width = bin_hi - bin_lo

    overlap = np.maximum(
        0.0,
        np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min),
    )

    weights = np.zeros_like(width, dtype=float)
    good = valid_ke & (width > 0)
    weights[good] = overlap[good] / width[good]
    return weights


def relative_change(y):
    """Expresses profile vectors as % variations relative to their mean."""
    y = np.asarray(y, dtype=float)
    mean = np.nanmean(y)
    with np.errstate(invalid="ignore", divide="ignore"):
        return 100.0 * (y - mean) / mean


def fit_relative_slope(x, y):
    """Executes simple standard linear regression over valid points."""
    y_rel = relative_change(y)
    good = np.isfinite(x) & np.isfinite(y_rel)
    if np.sum(good) < 3:
        return None
    return linregress(x[good], y_rel[good])


# Target normalisation parameters
w_norm_gt50 = ke_window_weights(NORMALISATION_MIN_KE_EV, np.inf)
electrons_gt50_z = np.nansum(counts_z_tof * w_norm_gt50[None, :], axis=1)

with np.errstate(invalid="ignore", divide="ignore"):
    # Normalisation 1: Traditional Electron-Yield Scheme
    spectrum_all_electrons = counts_z_tof / total_electrons_z[:, None]

    # Normalisation 2: Low-Energy Filtered Electron Yield
    spectrum_gt50 = counts_z_tof / electrons_gt50_z[:, None]

    # Normalisation 3: Shot-Normalized Absolute Yield
    spectrum_per_shot = counts_z_tof / shots_z[:, None]

spectra = {
    "all electrons": spectrum_all_electrons,
    f"electrons > {NORMALISATION_MIN_KE_EV:.0f} eV": spectrum_gt50,
    "per shot": spectrum_per_shot,
}

# Generate relative deviation arrays
relative_maps = {}
for name, spectrum in spectra.items():
    avg_spectrum = np.nanmean(spectrum, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        relative_maps[name] = (
            100.0 * (spectrum - avg_spectrum[None, :]) / avg_spectrum[None, :]
        )

# Isolate specific window constraints (220 eV - 270 eV)
plot_mask = (
    valid_ke & (ke_cent >= PLOT_KE_RANGE_EV[0]) & (ke_cent <= PLOT_KE_RANGE_EV[1])
)
idx = np.where(plot_mask)[0]

if idx.size == 0:
    raise ValueError(f"No calibrated KE bins inside {PLOT_KE_RANGE_EV}.")

lo_i, hi_i = int(idx[0]), int(idx[-1]) + 1
ke_edges_crop = ke_edges[lo_i : hi_i + 1]

maps_crop = {name: rel_map[:, lo_i:hi_i] for name, rel_map in relative_maps.items()}

# Ensure standard monotonic tracking order along the physical X axis
if ke_edges_crop[0] > ke_edges_crop[-1]:
    ke_edges_plot = ke_edges_crop[::-1]
    maps_crop = {name: m[:, ::-1] for name, m in maps_crop.items()}
else:
    ke_edges_plot = ke_edges_crop

# Establish stable symmetric scaling using 98th percentile boundaries
finite_maps = [m[np.isfinite(m)] for m in maps_crop.values() if m[np.isfinite(m)].size]
vmax = (
    np.nanpercentile(np.abs(np.concatenate(finite_maps)), 98) if finite_maps else 10.0
)
if not np.isfinite(vmax) or vmax <= 0:
    vmax = 10.0

# FIGURE 1: 2D Relative Change Maps
normalisation_names = list(spectra.keys())
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True, constrained_layout=True)

for ax, name in zip(axes, normalisation_names):
    pcm = ax.pcolormesh(
        ke_edges_plot,
        delay_edges_fs,
        maps_crop[name],
        shading="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_xlim(PLOT_KE_RANGE_EV)
    ax.set_xlabel("electron kinetic energy (eV)")
    ax.set_title(name)

axes[0].set_ylabel("delay (fs)")
fig.colorbar(pcm, ax=axes, label="relative change (%)")
fig.suptitle(
    f"{SCAN_NAME}\nEffect of normalisation on the delay-dependent spectral gradient",
    fontsize=13,
)
plt.show()

# Integrated Thesis Energy Windows Analysis
w_low = ke_window_weights(*LOW_WINDOW_KE_EV)
w_high = ke_window_weights(*HIGH_WINDOW_KE_EV)

C_low = np.nansum(counts_z_tof * w_low[None, :], axis=1)
C_high = np.nansum(counts_z_tof * w_high[None, :], axis=1)

trace_results = {}
with np.errstate(invalid="ignore", divide="ignore"):
    trace_results["all electrons"] = {
        "low": C_low / total_electrons_z,
        "high": C_high / total_electrons_z,
    }
    trace_results[f"electrons > {NORMALISATION_MIN_KE_EV:.0f} eV"] = {
        "low": C_low / electrons_gt50_z,
        "high": C_high / electrons_gt50_z,
    }
    trace_results["per shot"] = {
        "low": C_low / shots_z,
        "high": C_high / shots_z,
    }

# FIGURE 2: Broad Thesis Windows Tracking
fig, axes = plt.subplots(
    1, 3, figsize=(15, 4.5), sharex=True, sharey=True, constrained_layout=True
)

for ax, name in zip(axes, normalisation_names):
    low_rel = relative_change(trace_results[name]["low"])
    high_rel = relative_change(trace_results[name]["high"])

    ax.plot(
        delay_fs,
        low_rel,
        "o-",
        ms=3,
        lw=1.0,
        label=f"low {LOW_WINDOW_KE_EV[0]:.0f}-{LOW_WINDOW_KE_EV[1]:.0f} eV",
    )
    ax.plot(
        delay_fs,
        high_rel,
        "o-",
        ms=3,
        lw=1.0,
        label=f"high {HIGH_WINDOW_KE_EV[0]:.0f}-{HIGH_WINDOW_KE_EV[1]:.0f} eV",
    )

    ax.axhline(0.0, color="k", lw=0.7, alpha=0.4)
    ax.set_title(name)
    ax.set_xlabel("delay (fs)")
    ax.grid(alpha=0.2)

axes[0].set_ylabel("relative yield change (%)")
axes[0].legend(fontsize=8)
fig.suptitle(
    f"{SCAN_NAME}\nBroad thesis windows under different normalisations", fontsize=13
)
plt.show()

# Diagnostic Numerical Summary Output
print("\n" + "=" * 70)
print("BROAD-WINDOW LINEAR SLOPES")
print("(relative change about each trace's mean; nothing is detrended)")
print("=" * 70)

for name in normalisation_names:
    low = trace_results[name]["low"]
    high = trace_results[name]["high"]

    low_fit = fit_relative_slope(delay_fs, low)
    high_fit = fit_relative_slope(delay_fs, high)

    print(f"\n[{name.upper()}]")
    print(
        f"  low  {LOW_WINDOW_KE_EV[0]:.0f}-{LOW_WINDOW_KE_EV[1]:.0f} eV: {low_fit.slope:+.4f} %/fs | r={low_fit.rvalue:+.3f}, p={low_fit.pvalue:.3g}"
    )
    print(
        f"  high {HIGH_WINDOW_KE_EV[0]:.0f}-{HIGH_WINDOW_KE_EV[1]:.0f} eV: {high_fit.slope:+.4f} %/fs | r={high_fit.rvalue:+.3f}, p={high_fit.pvalue:.3g}"
    )

print("\n" + "=" * 70)
print("NORMALISATION DENOMINATORS BEHAVIOUR")
print("=" * 70)

denominators_list = [
    ("all detected electrons / shot", total_electrons_z / shots_z),
    (
        f"electrons > {NORMALISATION_MIN_KE_EV:.0f} eV / shot",
        electrons_gt50_z / shots_z,
    ),
]

for label, denominator in denominators_list:
    fit = fit_relative_slope(delay_fs, denominator)
    print(f"  {label}: {fit.slope:+.4f} %/fs | r={fit.rvalue:+.3f}, p={fit.pvalue:.3g}")
print("=" * 70 + "\n")


### Delay traces across incident photon energy

In [ ]:
# Compare these incident photon energies.
DIAG_ENERGIES_EV = [
    272.0,
    272.5,
    273.0,
    273.5,
    274.0,
    274.5,
    275.0,
    275.5,
    276.0,
    276.5,
]
LOW_WINDOW_KE_EV = (225.0, 240.0)
HIGH_WINDOW_KE_EV = (245.0, 266.0)
DELAY_ROUND_DECIMALS = 6


def _ke_window_weights(tof_edges, ke_min, ke_max):
    tof_edges = np.asarray(tof_edges, dtype=float)
    ke_edges = tof_to_ke(tof_edges)
    e0, e1 = ke_edges[:-1], ke_edges[1:]

    valid = np.isfinite(e0) & np.isfinite(e1)
    bin_lo, bin_hi = np.minimum(e0, e1), np.maximum(e0, e1)
    bin_width = bin_hi - bin_lo
    overlap = np.maximum(0.0, np.minimum(bin_hi, ke_max) - np.maximum(bin_lo, ke_min))

    weights = np.zeros_like(bin_width, dtype=float)
    good = valid & (bin_width > 0)
    weights[good] = overlap[good] / bin_width[good]
    return weights


def _extract_diag_from_agg(a):
    if a.D is None:
        raise ValueError("Aggregate has no eTOF dataset D.")
    if a.z_edges is None:
        raise ValueError("Aggregate is not time-resolved.")

    D = np.asarray(a.D, dtype=float)  # (gmd, delay, tof)
    G = np.asarray(a.G, dtype=float)  # (gmd, delay)
    n = np.asarray(a.n_per_bin, dtype=float)  # (gmd, delay)

    if D.ndim != 3:
        raise ValueError(f"Expected D to have 3 dimensions, got {D.shape}")
    if D.shape[:2] != n.shape:
        raise ValueError(
            f"D leading dimensions {D.shape[:2]} do not match n_per_bin shape {n.shape}"
        )
    if G.shape != n.shape:
        raise ValueError(f"G shape {G.shape} does not match n_per_bin shape {n.shape}")
    if D.shape[2] != len(a.tof_edges) - 1:
        raise ValueError(
            f"D has {D.shape[2]} TOF bins but tof_edges defines {len(a.tof_edges) - 1}"
        )

    counts_z_tof = np.nansum(D * n[..., None], axis=0)
    shots_z = np.nansum(n, axis=0)
    gmd_sum_z = np.nansum(G * n, axis=0)
    total_electrons_z = np.nansum(counts_z_tof, axis=1)

    w_low = _ke_window_weights(a.tof_edges, LOW_WINDOW_KE_EV[0], LOW_WINDOW_KE_EV[1])
    w_high = _ke_window_weights(a.tof_edges, HIGH_WINDOW_KE_EV[0], HIGH_WINDOW_KE_EV[1])
    low_counts_z = np.nansum(counts_z_tof * w_low[None, :], axis=1)
    high_counts_z = np.nansum(counts_z_tof * w_high[None, :], axis=1)

    z_edges = np.asarray(a.z_edges, dtype=float)
    delay_fs = (
        0.5 * (z_edges[:-1] + z_edges[1:]) - DELAY_ZERO_STAGE
    ) * FS_PER_STAGE_UNIT

    return {
        "delay_fs": delay_fs,
        "low_counts": low_counts_z,
        "high_counts": high_counts_z,
        "total_electrons": total_electrons_z,
        "gmd_sum": gmd_sum_z,
        "shots": shots_z,
    }


def _combine_diag_records(records):
    combined = {}
    for record in records:
        for i, delay in enumerate(record["delay_fs"]):
            key = round(float(delay), DELAY_ROUND_DECIMALS)
            if key not in combined:
                combined[key] = {
                    "low_counts": 0.0,
                    "high_counts": 0.0,
                    "total_electrons": 0.0,
                    "gmd_sum": 0.0,
                    "shots": 0.0,
                }

            for field in combined[key]:
                combined[key][field] += record[field][i]

    delays = np.array(sorted(combined), dtype=float)
    out = {"delay_fs": delays}
    for name in ["low_counts", "high_counts", "total_electrons", "gmd_sum", "shots"]:
        out[name] = np.array([combined[d][name] for d in delays], dtype=float)
    return out


def _finish_diag_quantities(record):
    total = record["total_electrons"]
    with np.errstate(invalid="ignore", divide="ignore"):
        low_yield, high_yield = (
            record["low_counts"] / total,
            record["high_counts"] / total,
        )

    low_mean, high_mean = np.nanmean(low_yield), np.nanmean(high_yield)
    with np.errstate(invalid="ignore", divide="ignore"):
        low_rel = 100.0 * (low_yield - low_mean) / low_mean
        high_rel = 100.0 * (high_yield - high_mean) / high_mean

    alpha = high_mean / low_mean
    with np.errstate(invalid="ignore", divide="ignore"):
        asymmetry_percent = (
            100.0 * (high_yield - alpha * low_yield) / (high_yield + alpha * low_yield)
        )

    record = dict(record)
    record.update(
        {
            "low_yield": low_yield,
            "high_yield": high_yield,
            "low_relative_percent": low_rel,
            "high_relative_percent": high_rel,
            "alpha": alpha,
            "asymmetry_percent": asymmetry_percent,
        }
    )
    return record


# Process and combine loaded files at each requested photon energy
diag_by_energy = {}
print("Files included in each photon-energy diagnostic:")

for energy in DIAG_ENERGIES_EV:
    matching = [
        (key, item)
        for key, item in loaded_aggs.items()
        if np.isclose(item["energy"], energy)
    ]
    if not matching:
        print(f"\n{energy:.1f} eV: no files")
        continue

    print(f"\n{energy:.1f} eV:")
    records = []
    for key, item in matching:
        print("   ", key)
        records.append(_extract_diag_from_agg(item["agg"]))

    combined = _finish_diag_quantities(_combine_diag_records(records))
    combined["source_keys"] = [key for key, _ in matching]
    diag_by_energy[energy] = combined

# Plot: rows = diagnostic quantities, columns = photon energies
energies_to_plot = [e for e in DIAG_ENERGIES_EV if e in diag_by_energy]
if not energies_to_plot:
    raise ValueError("None of DIAG_ENERGIES_EV were found in loaded_aggs.")

ncols = len(energies_to_plot)
fig, axes = plt.subplots(
    6,
    ncols,
    figsize=(3.4 * ncols, 13),
    sharex="col",
    squeeze=False,
    constrained_layout=True,
)
row_labels = [
    f"high {HIGH_WINDOW_KE_EV[0]:.0f}-{HIGH_WINDOW_KE_EV[1]:.0f} eV\nrelative yield (%)",
    f"low {LOW_WINDOW_KE_EV[0]:.0f}-{LOW_WINDOW_KE_EV[1]:.0f} eV\nrelative yield (%)",
    "balanced high-low\nasymmetry (%)",
    "total detected\nelectrons",
    "summed GMD",
    "shots",
]

for col, energy in enumerate(energies_to_plot):
    r, t = diag_by_energy[energy], diag_by_energy[energy]["delay_fs"]

    axes[0, col].plot(t, r["high_relative_percent"], "o-", ms=2.5, lw=0.9)
    axes[1, col].plot(t, r["low_relative_percent"], "o-", ms=2.5, lw=0.9)
    axes[2, col].plot(t, r["asymmetry_percent"], "o-", ms=2.5, lw=0.9)
    axes[3, col].plot(t, r["total_electrons"], "o-", ms=2.5, lw=0.9)
    axes[4, col].plot(t, r["gmd_sum"], "o-", ms=2.5, lw=0.9)
    axes[5, col].plot(t, r["shots"], "o-", ms=2.5, lw=0.9)

    for r_idx in range(3):
        axes[r_idx, col].axhline(0.0, color="k", lw=0.7, alpha=0.4)
    axes[0, col].set_title(
        f"{energy:.1f} eV\n{len(r['source_keys'])} file{'s' if len(r['source_keys']) != 1 else ''}"
    )

    for row in range(6):
        axes[row, col].grid(alpha=0.2)
    axes[5, col].set_xlabel("delay (fs)")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label)
fig.suptitle(
    "Delay-scan diagnostics by incident photon energy\n(repeated runs combined; no residual-gas subtraction)",
    fontsize=13,
)
plt.show()


### Separate-run comparison at one photon energy

In [ ]:
COMPARE_ENERGY_EV = 274.0

matching_runs = [
    (key, item)
    for key, item in loaded_aggs.items()
    if np.isclose(item["energy"], COMPARE_ENERGY_EV)
]
if len(matching_runs) < 2:
    raise ValueError(
        f"Only {len(matching_runs)} file(s) found at {COMPARE_ENERGY_EV:.1f} eV."
    )


def _short_run_label(key):
    label = Path(key).stem
    for text in ["_aggregates_tr_etof1", "_aggregates_tr", "_aggregates"]:
        label = label.replace(text, "")
    return label.replace("glycine_delay_scan_", "")


run_diagnostics = {}
print(f"Comparing runs at {COMPARE_ENERGY_EV:.1f} eV:")
for key, item in matching_runs:
    print("  ", key)
    run_diagnostics[key] = _finish_diag_quantities(_extract_diag_from_agg(item["agg"]))

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True, constrained_layout=True)
axes = axes.ravel()

for key, result in run_diagnostics.items():
    t, label = result["delay_fs"], _short_run_label(key)
    axes[0].plot(t, result["high_relative_percent"], "o-", ms=3, lw=1.0, label=label)
    axes[1].plot(t, result["low_relative_percent"], "o-", ms=3, lw=1.0, label=label)
    axes[2].plot(t, result["asymmetry_percent"], "o-", ms=3, lw=1.0, label=label)
    axes[3].plot(t, result["total_electrons"], "o-", ms=3, lw=1.0, label=label)
    axes[4].plot(t, result["gmd_sum"], "o-", ms=3, lw=1.0, label=label)
    axes[5].plot(t, result["shots"], "o-", ms=3, lw=1.0, label=label)

# Subplot labels and decoration
axes[0].set_title(
    f"High region: {HIGH_WINDOW_KE_EV[0]:.0f}-{HIGH_WINDOW_KE_EV[1]:.0f} eV"
)
axes[0].set_ylabel("relative electron yield (%)")
axes[1].set_title(f"Low region: {LOW_WINDOW_KE_EV[0]:.0f}-{LOW_WINDOW_KE_EV[1]:.0f} eV")
axes[1].set_ylabel("relative electron yield (%)")
axes[2].set_title("Balanced high-low asymmetry")
axes[2].set_ylabel("asymmetry (%)")
axes[3].set_title("Total detected electrons")
axes[3].set_ylabel("electrons")
axes[4].set_title("Summed GMD")
axes[4].set_ylabel("summed GMD")
axes[5].set_title("Shots per delay")
axes[5].set_ylabel("shots")

for i in [0, 1, 2]:
    axes[i].axhline(0.0, color="k", lw=0.7, alpha=0.4)
for ax in axes:
    ax.set_xlabel("delay (fs)")
    ax.grid(alpha=0.2)
axes[0].legend(title=f"{COMPARE_ENERGY_EV:.1f} eV runs", fontsize=8)

fig.suptitle(
    f"Run-to-run comparison at {COMPARE_ENERGY_EV:.1f} eV\n(no residual-gas subtraction)",
    fontsize=13,
)
plt.show()


## Notes

The single pipeline keeps the four 274 eV runs separate and uses a fixed-period fit, so agreement between fitted phases is a run-to-run consistency diagnostic rather than an unconstrained period measurement. The gradient diagnostics include both aggregate-level and shot-resolved checks because acquisition-order effects cannot be recovered from a fully merged delay aggregate. Static residual-gas subtraction cannot remove a genuinely delay-dependent background contribution.